In [1]:
# ═══════════════════════════════════════════════════════════════════════════

# ═══════════════════════════════════════════════════════════════════════════

# ── GOOGLE DRIVE SETUP ───────────────────────────────────────────────────
from google.colab import drive as _gdrive
# _gdrive.mount("/content/drive", force_remount=False)
import os as _os
DRIVE_CKPT_DIR      = "/content/pinn_checkpoints"
DRIVE_CKPT_INTERVAL = 2000
_os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print(f"[Drive] Checkpoint directory : {DRIVE_CKPT_DIR}")
print(f"[Drive] Periodic interval    : every {DRIVE_CKPT_INTERVAL} epochs (stage1)")
# ─────────────────────────────────────────────────────────────────────────

import os, warnings, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as _gs
import matplotlib.patches as _mp
from collections import defaultdict
from torch.autograd import grad as autograd_grad
from scipy.stats import mannwhitneyu
warnings.filterwarnings("ignore")

SEED = 42

def set_seed(s):
    torch.manual_seed(s); np.random.seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] {device}")

# ═══════════════════════════════════════════════════════════════════════════
#  MEASURED SITE PARAMETERS
# ═══════════════════════════════════════════════════════════════════════════

X_MAX        = 52.0
T_MAX_FULL   = 2114.0
T_MAX_SYNC   = 1596.0

# CHG-1: 80/10/10 split — T_MAX_TRAIN covers first 80% of synchronised record
T_MAX_TRAIN  = T_MAX_SYNC   # normalisation base = training horizon
T_MAX        = T_MAX_FULL

G_ACC        = 9.81
RHO_W        = 1000.0
GAMMA_W      = RHO_W * G_ACC

SITE = {
    107: dict(
        x_pos=25.0, sensor_depth_m=0.22, H_m=0.28, slope_deg=24.3,
        failure_depth_m=1.0,
        theta_r=0.100, theta_s=0.420,
        alpha=2.70, n_vg=1.23, Ks=2.89e-7,
        rho_b=1643.0,
        c_prime=2.0,
        phi_prime=30.0, label="Sandy Clay",
    ),
    108: dict(
        x_pos=7.0, sensor_depth_m=0.30, H_m=0.42, slope_deg=19.86,
        failure_depth_m=1.0,
        theta_r=0.063, theta_s=0.450,
        alpha=2.10, n_vg=1.48, Ks=3.64e-6,
        rho_b=1616.0,
        c_prime=1.0,
        phi_prime=30.0, label="Sandy Clay Loam",
    ),
}

X_POS_107   = SITE[107]["x_pos"]
X_POS_108   = SITE[108]["x_pos"]
X_NORM_107  = X_POS_107 / X_MAX
X_NORM_108  = X_POS_108 / X_MAX
Z_MAX       = max(SITE[107]["H_m"], SITE[108]["H_m"])

N_HIDDEN  = 4
N_WIDTH   = 96
DROPOUT   = 0.00

CSV_FILE        = "/content/2d_pinn_data_dev_107_108.csv"

# CHG-3: DRY_STEP = 10 for BOTH devices (more dry-period coverage)
DOWNSAMPLE_STEP = 15
DRY_STEP_108    = 15     # was 5
DRY_STEP_107    = 15     # was 20

SHARED_T0       = 4.58
GAP_THRESH_H    = 2.0
THETA_LO_107 = SITE[107]['theta_r']
THETA_HI_107 = SITE[107]['theta_s']
THETA_LO_108 = SITE[108]['theta_r']
THETA_HI_108 = SITE[108]['theta_s']
RAINFALL_UNIT   = "mm/hr"
RAIN_GATE_MS    = 1e-7

# CHG-1: 80/10/10 chronological split boundaries (as fractions of T_MAX_SYNC)
#   Train : [0,   0.80) × T_MAX_SYNC
#   Val   : [0.80, 0.90) × T_MAX_SYNC
#   Test  : [0.90, 1.00] × T_MAX_SYNC  ← strictly unseen
VAL_LO = 0.53   # val window start  (was 0.35 middle-holdout)
VAL_HI = 0.63   # val window end    (was 0.65)
TE_LO  = 0.90   # test window start (was 0.85)

PTF_BOUNDS = {
    107: dict(
        alpha  =(2.70*0.80, 2.70*1.20),
        n_vg   =(1.23*0.80, 1.23*1.20),
        theta_r=(0.100-0.030, 0.100+0.030),
        theta_s=(0.350, 0.550),
    Ks     =(SITE[107]["Ks"] * 0.1, SITE[107]["Ks"] * 10.0), # Added Ks bounds
    ),
    108: dict(
        alpha  =(5.90*0.70, 5.90*1.30),
        n_vg   =(1.48*0.70, 1.48*1.30),
        theta_r=(0.063-0.026, 0.063+0.026),
        theta_s=(0.300, 0.580),
    Ks     =(SITE[108]["Ks"] * 0.1, SITE[108]["Ks"] * 10.0), # Added Ks bounds
    ),
}


# ═══════════════════════════════════════════════════════════════════════════
#  CHG-1 — CHRONOLOGICAL 80/10/10 SPLIT
# ═══════════════════════════════════════════════════════════════════════════

def _splits(t_h):
    """
    Train : t_norm in [0, 53%) AND [63%, 90%)
    Val   : t_norm in [53%, 63%)
    Test  : t_norm in [90%, 100%]
    """
    t_clip = np.minimum(t_h, T_MAX_SYNC)
    tn     = t_clip / T_MAX_SYNC

    val_mask = (tn >= VAL_LO) & (tn < VAL_HI)             # [53%, 63%)
    te_mask  = (tn >= TE_LO)  & (t_h <= T_MAX_SYNC)       # [90%, 100%]

    # Train is everything before the Test set, minus the Val set
    tr_mask  = (tn < TE_LO) & ~val_mask     # [90%, 100%] — unseen

    return tr_mask, val_mask, te_mask


# ═══════════════════════════════════════════════════════════════════════════
#  PARAMETER INTERPOLATION
# ═══════════════════════════════════════════════════════════════════════════

def _interp_x(x_norm, val107, val108):
    xn107 = X_NORM_107; xn108 = X_NORM_108
    t = torch.clamp((x_norm - xn108) / (xn107 - xn108 + 1e-9), 0.0, 1.0)
    return (1.0 - t) * val108 + t * val107

def get_H(x_norm):
    return _interp_x(x_norm,
                     torch.tensor(SITE[107]["H_m"], dtype=torch.float32),
                     torch.tensor(SITE[108]["H_m"], dtype=torch.float32))

def get_failure_depth(x_norm):
    return _interp_x(x_norm,
                     torch.tensor(SITE[107]["failure_depth_m"], dtype=torch.float32),
                     torch.tensor(SITE[108]["failure_depth_m"], dtype=torch.float32))

def get_slope_rad(x_norm):
    b107 = np.radians(SITE[107]["slope_deg"])
    b108 = np.radians(SITE[108]["slope_deg"])
    return _interp_x(x_norm,
                     torch.tensor(b107, dtype=torch.float32),
                     torch.tensor(b108, dtype=torch.float32))

def get_rho_b(x_norm):
    return _interp_x(x_norm,
                     torch.tensor(SITE[107]["rho_b"], dtype=torch.float32),
                     torch.tensor(SITE[108]["rho_b"], dtype=torch.float32))

def get_phi_rad(x_norm):
    p107 = np.radians(SITE[107]["phi_prime"])
    p108 = np.radians(SITE[108]["phi_prime"])
    return _interp_x(x_norm,
                     torch.tensor(p107, dtype=torch.float32),
                     torch.tensor(p108, dtype=torch.float32))

def get_cohesion(x_norm):
    return _interp_x(x_norm,
                     torch.tensor(SITE[107]["c_prime"] * 1000.0, dtype=torch.float32),
                     torch.tensor(SITE[108]["c_prime"] * 1000.0, dtype=torch.float32))


# ═══════════════════════════════════════════════════════════════════════════
#  VAN GENUCHTEN
# ═══════════════════════════════════════════════════════════════════════════

def vg_theta(psi, x_norm, params=None):
    if params is not None:
        alpha   = _interp_x(x_norm, params["alpha"][0],   params["alpha"][1])
        n       = _interp_x(x_norm, params["n_vg"][0],    params["n_vg"][1])
        theta_r = _interp_x(x_norm, params["theta_r"][0], params["theta_r"][1])
        theta_s = _interp_x(x_norm, params["theta_s"][0], params["theta_s"][1])
    else:
        alpha   = _interp_x(x_norm,
                    torch.tensor(SITE[107]["alpha"],   dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["alpha"],   dtype=torch.float32, device=psi.device))
        n       = _interp_x(x_norm,
                    torch.tensor(SITE[107]["n_vg"],    dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["n_vg"],    dtype=torch.float32, device=psi.device))
        theta_r = _interp_x(x_norm,
                    torch.tensor(SITE[107]["theta_r"], dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["theta_r"], dtype=torch.float32, device=psi.device))
        theta_s = _interp_x(x_norm,
                    torch.tensor(SITE[107]["theta_s"], dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["theta_s"], dtype=torch.float32, device=psi.device))
    m   = 1.0 - 1.0 / n
    arg = torch.clamp(alpha * torch.abs(psi), min=0.0)
    Se  = 1.0 / (1.0 + arg.pow(n)).pow(m)
    Se  = torch.where(psi >= 0.0, torch.ones_like(psi), Se)
    Se  = torch.clamp(Se, 1e-6, 1.0 - 1e-6)
    return theta_r + (theta_s - theta_r) * Se

def vg_Se(psi, x_norm, params=None):
    if params is not None:
        alpha = _interp_x(x_norm, params["alpha"][0], params["alpha"][1])
        n     = _interp_x(x_norm, params["n_vg"][0],  params["n_vg"][1])
    else:
        alpha = _interp_x(x_norm,
                    torch.tensor(SITE[107]["alpha"], dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["alpha"], dtype=torch.float32, device=psi.device))
        n     = _interp_x(x_norm,
                    torch.tensor(SITE[107]["n_vg"],  dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["n_vg"],  dtype=torch.float32, device=psi.device))
    m   = 1.0 - 1.0 / n
    arg = torch.clamp(alpha * torch.abs(psi), min=0.0)
    Se  = 1.0 / (1.0 + arg.pow(n)).pow(m)
    Se  = torch.where(psi >= 0.0, torch.ones_like(psi), Se)
    return torch.clamp(Se, 1e-6, 1.0)

def vg_K(psi, x_norm, params=None):
    if params is not None:
        alpha = _interp_x(x_norm, params["alpha"][0], params["alpha"][1])
        n     = _interp_x(x_norm, params["n_vg"][0],  params["n_vg"][1])
        Ks    = _interp_x(x_norm, params["Ks"][0],    params["Ks"][1])
    else:
        alpha = _interp_x(x_norm,
                    torch.tensor(SITE[107]["alpha"], dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["alpha"], dtype=torch.float32, device=psi.device))
        n     = _interp_x(x_norm,
                    torch.tensor(SITE[107]["n_vg"],  dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["n_vg"],  dtype=torch.float32, device=psi.device))
        Ks    = _interp_x(x_norm,
                    torch.tensor(SITE[107]["Ks"],    dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["Ks"],    dtype=torch.float32, device=psi.device))
    m   = 1.0 - 1.0 / n
    arg = torch.clamp(alpha * torch.abs(psi), min=0.0)
    Se  = 1.0 / (1.0 + arg.pow(n)).pow(m)
    Se  = torch.where(psi >= 0.0, torch.ones_like(psi), Se)
    Se  = torch.clamp(Se, 1e-6, 1.0 - 1e-6)
    inner = torch.clamp(1.0 - torch.clamp(Se, 1e-6, 1-1e-6).pow(1.0/m), min=0.0)
    K = Ks * Se.pow(0.5) * (1.0 - inner.pow(m)).pow(2.0)
    return torch.clamp(K, 1e-15, 1e-2)

def dtheta_dpsi(psi, x_norm, params=None):
    if params is not None:
        alpha   = _interp_x(x_norm, params["alpha"][0],   params["alpha"][1])
        n       = _interp_x(x_norm, params["n_vg"][0],    params["n_vg"][1])
        theta_r = _interp_x(x_norm, params["theta_r"][0], params["theta_r"][1])
        theta_s = _interp_x(x_norm, params["theta_s"][0], params["theta_s"][1])
    else:
        alpha   = _interp_x(x_norm,
                    torch.tensor(SITE[107]["alpha"],   dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["alpha"],   dtype=torch.float32, device=psi.device))
        n       = _interp_x(x_norm,
                    torch.tensor(SITE[107]["n_vg"],    dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["n_vg"],    dtype=torch.float32, device=psi.device))
        theta_r = _interp_x(x_norm,
                    torch.tensor(SITE[107]["theta_r"], dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["theta_r"], dtype=torch.float32, device=psi.device))
        theta_s = _interp_x(x_norm,
                    torch.tensor(SITE[107]["theta_s"], dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["theta_s"], dtype=torch.float32, device=psi.device))
    m    = 1.0 - 1.0 / n
    dts  = theta_s - theta_r
    arg  = torch.clamp(alpha * torch.abs(psi), min=0.0)
    C    = dts * m * n * alpha * arg.pow(n - 1.0) / (1.0 + arg.pow(n)).pow(m + 1.0)
    C    = torch.where(psi >= 0.0, torch.zeros_like(psi), C)
    return torch.clamp(C, 0.0, 10.0)


# ═══════════════════════════════════════════════════════════════════════════
#  PORE PRESSURE & FACTOR OF SAFETY
# ═══════════════════════════════════════════════════════════════════════════

def compute_pore_pressure(psi, x_norm, theta, params=None):
    """
    Extended pore pressure (FIX-B): allows u > 0 above θₛ (overpressure).
    Unsaturated: u = Se × γ_w × ψ   (ψ<0 → suction, stabilising)
    Saturated  : u = γ_w × ψ         (ψ≥0 → positive pressure, destabilising)
    """
    gw  = torch.tensor(GAMMA_W, dtype=torch.float32, device=psi.device)
    Se  = vg_Se(psi, x_norm, params)
    u_unsat = Se * gw * psi
    u_sat   = gw * psi
    u = torch.where(psi >= 0.0, u_sat, u_unsat)
    return torch.clamp(u, min=-2e5, max=2e5)


def fos_infinite_slope(psi, x_norm, theta=None, params=None):
    """
    CHG-4: PRIMARY OUTPUT — Factor of Safety (infinite slope, Bishop).

    FoS = [c′ + (γₛ·z_f·cos²β − u) · tan φ′] / [γₛ·z_f·sinβ·cosβ]

    z_f  = failure plane depth = 1.0 m  (FIX-F)
    u    = pore water pressure            (FIX-B: positive above θₛ)
    c′   = effective cohesion 1–2 kPa    (FIX-F)
    """
    z_f     = get_failure_depth(x_norm).to(psi.device)
    beta    = get_slope_rad(x_norm).to(psi.device)
    rho_b   = get_rho_b(x_norm).to(psi.device)
    phi_p   = get_phi_rad(x_norm).to(psi.device)
    c_prime = get_cohesion(x_norm).to(psi.device)   # [Pa]
    gamma_s = rho_b * G_ACC

    u_w     = compute_pore_pressure(psi, x_norm, theta, params)

    sigma_n     = gamma_s * z_f * torch.cos(beta)**2
    sigma_n_eff = torch.clamp(sigma_n - u_w, min=0.0)
    tau_d       = gamma_s * z_f * torch.sin(beta) * torch.cos(beta) + 1e-3

    fos = torch.clamp(
        (c_prime + sigma_n_eff * torch.tan(phi_p)) / tau_d,
        0.05, 15.0
    )
    return fos


# ═══════════════════════════════════════════════════════════════════════════
#  VALIDATED FoS SPATIAL SCAN
# ═══════════════════════════════════════════════════════════════════════════

def _fos_analytical_floor(x_norm_val):
    """Analytical FoS floor at ψ=0 (fully saturated). Used for validation."""
    beta = float(np.radians(
        SITE[107]["slope_deg"] * float(x_norm_val) +
        SITE[108]["slope_deg"] * (1 - float(x_norm_val))
    ))
    phi      = float(np.radians(SITE[107]["phi_prime"]))
    z_f      = SITE[107]["failure_depth_m"]
    gamma_s  = SITE[107]["rho_b"] * G_ACC
    c_prime  = SITE[107]["c_prime"] * 1000.0

    sigma_n  = gamma_s * z_f * np.cos(beta)**2
    u_max    = GAMMA_W * z_f
    tau_d    = gamma_s * z_f * np.sin(beta) * np.cos(beta) + 1e-3
    fos_sat  = (c_prime + max(sigma_n - u_max, 0) * np.tan(phi)) / tau_d
    fos_dry  = (c_prime + sigma_n * np.tan(phi)) / tau_d
    return float(fos_sat), float(fos_dry)


def fos_spatial_scan_validated(model, t_mid_h, n_scan=20):
    """FIX-D: Spatial FoS scan with validation against saturated analytical floor."""
    x_scan = torch.linspace(X_NORM_108, X_NORM_107, n_scan, device=device).unsqueeze(1)
    z_b    = torch.ones(n_scan, 1, device=device)
    t_s    = torch.full((n_scan, 1), t_mid_h / T_MAX_TRAIN, device=device)

    with torch.no_grad():
        psi_scan, _, fos_scan_raw = model(x_scan, z_b, t_s)

    fos_np = fos_scan_raw.cpu().numpy().flatten()
    x_np   = x_scan.cpu().numpy().flatten()

    fos_min_conservative = float(fos_np.min()) * 0.95

    x_mid = float(np.mean([X_NORM_107, X_NORM_108]))
    fos_sat_floor, fos_dry_floor = _fos_analytical_floor(x_mid)

    validated   = fos_min_conservative >= fos_sat_floor * 0.90
    val_status  = "validated" if validated else "⚠ BELOW SAT FLOOR — check params"

    return {
        "fos_min_interp" : fos_min_conservative,
        "fos_scan"       : fos_np,
        "x_scan_norm"    : x_np,
        "fos_sat_floor"  : fos_sat_floor,
        "fos_dry_floor"  : fos_dry_floor,
        "validated"      : validated,
        "val_status"     : val_status,
    }


# ═══════════════════════════════════════════════════════════════════════════
#  PINN MODEL
# ═══════════════════════════════════════════════════════════════════════════

class PINNSlope(nn.Module):
    """
    CHG-4: model.forward() returns (ψ [m], θ [m³/m³], FoS [-]).
    FoS is the primary output; ψ and θ are intermediate physical states.
    UPGRADE: Multiplicative Filter Network (Modified MLP) with Fourier Features
    to cure Spectral Bias and fit sharp rainfall spikes.
    """
    def __init__(self, hidden=N_HIDDEN, width=N_WIDTH, dropout=DROPOUT,
                 ts107_init=None, ts108_init=None):
        super().__init__()

        # --- FOURIER FEATURE EMBEDDING SETUP ---
        self.n_freqs = 8  # Increased from 6 to 10 for sharper spikes
        # Input: x(1) + z(1) + t(1) + sin_t(10) + cos_t(10) = 23 dimensions
        in_dim = 3 + (2 * self.n_freqs)

        # 1. Project inputs into two parallel gating feature spaces (U and V)
        self.U = nn.Sequential(nn.Linear(in_dim, width), nn.Tanh())
        self.V = nn.Sequential(nn.Linear(in_dim, width), nn.Tanh())

        # 2. Setup the hidden layers as a ModuleList
        self.first_layer = nn.Linear(in_dim, width)
        self.hidden_layers = nn.ModuleList(
            [nn.Linear(width, width) for _ in range(hidden - 1)]
        )

        self.head_psi = nn.Linear(width, 1)

        # --- PHYSICAL PARAMETERS ---
        self._log_alpha107 = nn.Parameter(torch.log(torch.tensor(SITE[107]["alpha"], dtype=torch.float32)))
        self._log_alpha108 = nn.Parameter(torch.log(torch.tensor(SITE[108]["alpha"], dtype=torch.float32)))
        self._raw_n107     = nn.Parameter(torch.tensor(SITE[107]["n_vg"],    dtype=torch.float32))
        self._raw_n108     = nn.Parameter(torch.tensor(SITE[108]["n_vg"],    dtype=torch.float32))
        self._theta_r107   = nn.Parameter(torch.tensor(SITE[107]["theta_r"], dtype=torch.float32))
        self._theta_r108   = nn.Parameter(torch.tensor(SITE[108]["theta_r"], dtype=torch.float32))

        ts107 = float(ts107_init) if ts107_init is not None else SITE[107]["theta_s"]
        ts108 = float(ts108_init) if ts108_init is not None else SITE[108]["theta_s"]
        self._theta_s107 = nn.Parameter(torch.tensor(ts107, dtype=torch.float32))
        self._theta_s108 = nn.Parameter(torch.tensor(ts108, dtype=torch.float32))
        self._log_Ks107 = nn.Parameter(torch.log(torch.tensor(SITE[107]["Ks"], dtype=torch.float32)))
        self._log_Ks108 = nn.Parameter(torch.log(torch.tensor(SITE[108]["Ks"], dtype=torch.float32)))
        self._init_weights()

    def _init_weights(self):
        # PyTorch's self.modules() automatically finds U, V, and hidden_layers
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight, gain=1.0)
                nn.init.zeros_(m.bias)

    @property
    def vg_params(self):
        def _clamp_log(param, lo, hi):
            return torch.exp(torch.clamp(param, np.log(lo), np.log(hi)))
        def _clamp(param, lo, hi):
            return torch.clamp(param, lo, hi)

        alpha107 = _clamp_log(self._log_alpha107, *PTF_BOUNDS[107]["alpha"])
        alpha108 = _clamp_log(self._log_alpha108, *PTF_BOUNDS[108]["alpha"])
        n107     = _clamp(self._raw_n107, *PTF_BOUNDS[107]["n_vg"])
        n108     = _clamp(self._raw_n108, *PTF_BOUNDS[108]["n_vg"])
        tr107    = _clamp(self._theta_r107, *PTF_BOUNDS[107]["theta_r"])
        tr108    = _clamp(self._theta_r108, *PTF_BOUNDS[108]["theta_r"])
        ts107    = _clamp(self._theta_s107, *PTF_BOUNDS[107]["theta_s"])
        ts108    = _clamp(self._theta_s108, *PTF_BOUNDS[108]["theta_s"])
        ts107    = torch.max(ts107, tr107.detach() + 0.05)
        ts108    = torch.max(ts108, tr108.detach() + 0.05)

        # --- NEW: Clamp Ks using the PTF bounds ---
        Ks107_learned = _clamp_log(self._log_Ks107, *PTF_BOUNDS[107]["Ks"])
        Ks108_learned = _clamp_log(self._log_Ks108, *PTF_BOUNDS[108]["Ks"])

        return dict(
            alpha  =(alpha107, alpha108),
            n_vg   =(n107,     n108),
            theta_r=(tr107,    tr108),
            theta_s=(ts107,    ts108),
            Ks     =(Ks107_learned, Ks108_learned), # Use the learned tensors
        )

    def get_learned_params(self):
        p = self.vg_params
        out = {}
        for k, (v107, v108) in p.items():
            out[f"{k}_107"] = float(v107.detach().cpu())
            out[f"{k}_108"] = float(v108.detach().cpu())
        return out

    def forward(self, x, z, t):
        # --- GENERATE FOURIER TIME FEATURES ---
        freqs = torch.pi * torch.pow(2.0, torch.arange(0, self.n_freqs, device=t.device))
        t_embed = t * freqs
        t_sin = torch.sin(t_embed)
        t_cos = torch.cos(t_embed)

        X_in = torch.cat([x, z, t, t_sin, t_cos], dim=1)

        # --- MULTIPLICATIVE MODULATION ---
        # 1. Generate the gates based on the raw, high-frequency inputs
        U_gate = self.U(X_in)
        V_gate = self.V(X_in)

        # 2. Process first layer
        h = torch.tanh(self.first_layer(X_in))

        # 3. Multiply through the deep hidden layers
        for layer in self.hidden_layers:
            z_feat = torch.tanh(layer(h))
            h = z_feat * U_gate + (1.0 - z_feat) * V_gate

        # --- FINAL OUTPUT HEAD ---
        raw   = self.head_psi(h)
        psi   = 8.0 * torch.tanh(raw) - 4.0

        p     = self.vg_params
        theta = vg_theta(psi, x, p)
        fos   = fos_infinite_slope(psi, x, theta=theta, params=p)
        return psi, theta, fos


# ═══════════════════════════════════════════════════════════════════════════
#  NaN-AWARE safe_sq
# ═══════════════════════════════════════════════════════════════════════════

_nan_counts   = defaultdict(int)
_total_counts = defaultdict(int)

def reset_nan_tracker():
    _nan_counts.clear(); _total_counts.clear()

def get_nan_fractions():
    return {tag: _nan_counts[tag] / max(_total_counts.get(tag, 1), 1)
            for tag in _nan_counts}

def safe_sq(r, tag=""):
    n_total     = r.numel()
    finite_mask = torch.isfinite(r)
    n_nan       = n_total - finite_mask.sum().item()
    _nan_counts[tag]   += n_nan
    _total_counts[tag] += n_total
    if n_nan > 0 and n_nan == n_total:
        return torch.tensor(0.0, device=r.device, requires_grad=False)
    r_clean  = torch.where(finite_mask, r, torch.zeros_like(r))
    r_clean  = torch.clamp(r_clean, -1e3, 1e3)
    n_finite = finite_mask.sum().float()
    v = (r_clean ** 2).sum() / (n_finite + 1e-9)
    return v if torch.isfinite(v) else torch.tensor(0.0, device=r.device, requires_grad=False)


# ═══════════════════════════════════════════════════════════════════════════
#  CAUSAL RAIN FUNCTION
# ═══════════════════════════════════════════════════════════════════════════

_rain_fn = None

def set_rain_fn(fn): global _rain_fn; _rain_fn = fn

def make_causal_rain_fn(t108_norm, q108, t107_norm, q107, xn108, xn107):
    t108_np = t108_norm.astype(np.float64); q108_np = q108.astype(np.float64)
    t107_np = t107_norm.astype(np.float64); q107_np = q107.astype(np.float64)

    def _rain_fn_impl(x_norm, t_norm):
        t_np = t_norm.detach().cpu().numpy().flatten().astype(np.float64)
        x_np = x_norm.detach().cpu().numpy().flatten().astype(np.float64)
        q8   = np.interp(t_np, t108_np, q108_np, left=0.0, right=float(q108_np[-1]))
        q7   = np.interp(t_np, t107_np, q107_np, left=0.0, right=float(q107_np[-1]))
        al   = np.clip((x_np - xn108) / (xn107 - xn108 + 1e-9), 0.0, 1.0)
        q    = np.clip((1 - al) * q8 + al * q7, 0.0, None)
        return torch.tensor(q, dtype=torch.float32, device=t_norm.device).reshape_as(t_norm)

    return _rain_fn_impl


# ═══════════════════════════════════════════════════════════════════════════
#  PHYSICS LOSSES
# ═══════════════════════════════════════════════════════════════════════════

def _grad(y, x):
    return autograd_grad(y, x, grad_outputs=torch.ones_like(y),
                         create_graph=True, retain_graph=True)[0]

def loss_richards(model, x, z, t):
    psi, _, _ = model(x, z, t)
    p              = model.vg_params
    C              = dtheta_dpsi(psi, x, p)
    K              = vg_K(psi, x, p)
    dpsi_dt_norm   = _grad(psi, t)
    dpsi_dz_norm   = _grad(psi, z)
    cos_beta       = torch.cos(get_slope_rad(x).to(psi.device))
    flux_z         = K * (dpsi_dz_norm / Z_MAX + cos_beta)
    dflux_dz       = _grad(flux_z, z) / Z_MAX
    T_train_s      = T_MAX_TRAIN * 3600.0
    residual       = C * dpsi_dt_norm / T_train_s - dflux_dz
    residual_scaled = residual * 1e6
    return safe_sq(residual_scaled, "richards")

def vg_Ks(x_norm, params=None):
    if params is not None:
        # Use the dynamically learned Ks values
        Ks107 = params["Ks"][0]
        Ks108 = params["Ks"][1]
    else:
        # Fallback to literature values
        Ks107 = torch.tensor(SITE[107]["Ks"], dtype=torch.float32, device=x_norm.device)
        Ks108 = torch.tensor(SITE[108]["Ks"], dtype=torch.float32, device=x_norm.device)

    alpha  = torch.clamp((x_norm - X_NORM_108) / (X_NORM_107 - X_NORM_108 + 1e-9), 0.0, 1.0)
    return Ks108 * (1.0 - alpha) + Ks107 * alpha

def loss_bc_top(model, x, z, t):
    psi, _, _ = model(x, z, t)
    if _rain_fn is None: return torch.tensor(0.0, device=x.device)
    p      = model.vg_params
    K      = vg_K(psi, x, p)
    q_rain = _rain_fn(x, t)
    Ks_x   = vg_Ks(x, p)
    gate   = (q_rain > RAIN_GATE_MS).float()
    if gate.sum().item() < 2:
        return torch.tensor(0.0, device=x.device, requires_grad=False)
    neumann_mask   = (q_rain <= Ks_x).float()
    dpsi_dz        = _grad(psi, z) / Z_MAX
    cos_beta       = torch.cos(get_slope_rad(x).to(psi.device))
    q_pred         = K * (dpsi_dz + cos_beta)
    Ks_ref         = 0.5 * (SITE[107]["Ks"] + SITE[108]["Ks"])
    L_neumann      = safe_sq(gate * neumann_mask * (q_pred - q_rain) / Ks_ref, "bc_top_neu")
    dirichlet_mask = (q_rain > Ks_x).float()
    L_dirichlet    = safe_sq(gate * dirichlet_mask * psi / 1.0, "bc_top_dir")
    return L_neumann + L_dirichlet

def loss_bc_bot(model, x, z, t):
    psi, _, _ = model(x, z, t)
    dpsi_dz = _grad(psi, z) / Z_MAX
    return safe_sq(dpsi_dz, "bc_bot")

PSI_IC_108 = -7.07
PSI_IC_107 = -10.0

def loss_ic(model, x, z, t):
    psi, _, _ = model(x, z, t)
    psi_ic_107 = torch.tensor(PSI_IC_107, dtype=torch.float32, device=x.device)
    psi_ic_108 = torch.tensor(PSI_IC_108, dtype=torch.float32, device=x.device)
    alpha_x    = torch.clamp((x - X_NORM_108) / (X_NORM_107 - X_NORM_108 + 1e-9), 0.0, 1.0)
    psi_ic     = psi_ic_108 * (1 - alpha_x) + psi_ic_107 * alpha_x
    psi_range  = float(abs(PSI_IC_107 - PSI_IC_108) + 1.0)
    return safe_sq((psi - psi_ic) / psi_range, "ic")

def loss_data(model, x_obs, z_obs, t_obs, theta_obs, q_rain_obs=None, event_weights=None):
    _, theta, _ = model(x_obs, z_obs, t_obs)
    residual = theta - theta_obs
    if event_weights is not None:
        return (event_weights * residual ** 2).mean()
    return safe_sq(residual, "data")

def loss_smoothness(model, x, z, t):
    psi, _, _ = model(x, z, t)
    dpsi_dx   = _grad(psi, x) / X_MAX
    d2psi_dx2 = _grad(dpsi_dx, x) / X_MAX
    return safe_sq(d2psi_dx2 * 0.3, "smooth")

def loss_psi_variance(model, x, z, t):
    psi, _, _ = model(x, z, t)
    psi_std   = psi.std()
    penalty   = torch.relu(0.5 - psi_std) ** 2
    return penalty

def loss_prior_vg(model):
    p    = model.vg_params
    loss = torch.tensor(0.0, device=next(model.parameters()).device)
    for key, (v107_ptf, v108_ptf) in [
        ("alpha",   (SITE[107]["alpha"],   SITE[108]["alpha"])),
        ("n_vg",    (SITE[107]["n_vg"],    SITE[108]["n_vg"])),
        ("theta_s", (SITE[107]["theta_s"], SITE[108]["theta_s"])),
    ]:
        p107 = torch.tensor(v107_ptf, dtype=torch.float32, device=loss.device)
        p108 = torch.tensor(v108_ptf, dtype=torch.float32, device=loss.device)
        loss = loss + 0.05 * ((p[key][0] - p107)/p107)**2
        loss = loss + 0.05 * ((p[key][1] - p108)/p108)**2
    return loss


def loss_fos_physics(model, x, z, t):
    """
    CHG-4/5: FoS physical regularisation.
    Penalises FoS below 0.5 (numerically implausible) or above 12 (geotechnically
    implausible for a rainfall-stressed slope). Guides the network to produce
    physically meaningful FoS throughout training — not just at sensor positions.
    """
    _, _, fos = model(x, z, t)
    # Soft lower bound: FoS should not collapse below 0.5
    penalty_lo  = torch.relu(0.5 - fos) ** 2
    # Soft upper bound: FoS should not exceed 12
    penalty_hi  = torch.relu(fos - 12.0) ** 2
    return safe_sq(penalty_lo + penalty_hi, "fos_physics")


# ═══════════════════════════════════════════════════════════════════════════
#  ADAPTIVE DOWNSAMPLING
# ═══════════════════════════════════════════════════════════════════════════

def _adaptive_ds(df_raw, rain_col="q_ms", dry_step=10):
    """
    CHG-3: Both devices use dry_step=10 (more dry-period coverage).
    Keeps ALL rainfall rows + every dry_step-th dry row.
    """
    rain_mask = df_raw[rain_col].values > 0
    dry_mask  = ~rain_mask

    wet_idx          = df_raw.index[rain_mask].tolist()
    dry_positions    = np.where(dry_mask)[0]
    dry_kept_pos     = dry_positions[::dry_step]
    dry_idx          = df_raw.index[dry_kept_pos].tolist()

    kept   = sorted(set(wet_idx) | set(dry_idx))
    df_out = df_raw.loc[kept].reset_index(drop=True)

    n_wet = len(wet_idx); n_dry = len(dry_idx)
    print(f"[AdaptiveDS] {rain_col[:5]}-device: "
          f"kept {n_wet} rain pts (ALL) + {n_dry} dry pts (step={dry_step}) "
          f"= {len(df_out)} total")
    return df_out


# ═══════════════════════════════════════════════════════════════════════════
#  DATA LOADER
# ═══════════════════════════════════════════════════════════════════════════

class SlopeDataLoader:
    def __init__(self, filepath, downsample=DOWNSAMPLE_STEP):
        self.filepath = filepath
        self.ds       = downsample
        self.ts107_raw_init = None
        self.ts108_raw_init = None

    def load(self):
        global T_MAX_SYNC, T_MAX_FULL, T_MAX, T_MAX_TRAIN

        raw = pd.read_csv(self.filepath)
        raw.columns = [c.strip().lower() for c in raw.columns]
        ts = pd.to_datetime(raw["timestamp"], dayfirst=False, errors="coerce")
        t0 = ts.min()
        raw["t_h"] = (ts - t0).dt.total_seconds() / 3600.0
        raw = raw[ts.notna()].copy()

        df108_raw = raw[raw["devid"]==108].copy().sort_values("t_h").reset_index(drop=True)
        df107_raw = raw[raw["devid"]==107].copy().sort_values("t_h").reset_index(drop=True)

        df108_raw = df108_raw[df108_raw["t_h"] >= SHARED_T0].copy().reset_index(drop=True)
        df107_raw = df107_raw[df107_raw["t_h"] >= SHARED_T0].copy().reset_index(drop=True)
        df108_raw["t_h"] -= SHARED_T0
        df107_raw["t_h"] -= SHARED_T0
        print(f"[StratB] t=0 reset to t_orig={SHARED_T0}h")

        T_MAX_SYNC  = float(df107_raw["t_h"].max())
        T_MAX_FULL  = float(np.ceil(df108_raw["t_h"].max() / 6.0) * 6.0)
        T_MAX       = T_MAX_FULL
        # CHG-1: training horizon = first 80% of synchronised record
        T_MAX_TRAIN = T_MAX_SYNC

        tr_hi_h  = T_MAX_SYNC * VAL_LO
        val_lo_h = T_MAX_SYNC * VAL_LO
        val_hi_h = T_MAX_SYNC * VAL_HI
        te_lo_h  = T_MAX_SYNC * TE_LO

        print(f"[CHG-1] 80/10/10 chronological split:")
        print(f"        Train : [0, {tr_hi_h:.0f}h)  —  T_MAX_TRAIN={T_MAX_TRAIN:.1f}h")
        print(f"        Val   : [{val_lo_h:.0f}h, {val_hi_h:.0f}h)  "
              f"({val_hi_h - val_lo_h:.0f}h window)")
        print(f"        Test  : [{te_lo_h:.0f}h, {T_MAX_SYNC:.0f}h]  "
              f"(STRICTLY UNSEEN — {T_MAX_SYNC - te_lo_h:.0f}h)")

        # ── FIX-A + FIX-C: rescaling ─────────────────────────────────────
        for df, did in [(df108_raw, 108), (df107_raw, 107)]:
            s = pd.to_numeric(df["soil"], errors="coerce").ffill().bfill()
            if s.max() > 1.0:
                s = s / 100.0

            theta_r_lit = SITE[did]["theta_r"]
            theta_s_lit = SITE[did]["theta_s"]

            valid    = (s >= theta_r_lit).values
            n_drop   = (~valid).sum()
            if n_drop > 0:
                print(f"[DataClean] Dev{did}: dropping {n_drop} rows "
                      f"with soil < {theta_r_lit}")
            df.drop(index=df.index[~valid], inplace=True)
            df.reset_index(drop=True, inplace=True)
            s_clean = s[valid].reset_index(drop=True)

            obs_min = float(s_clean.min()); obs_max = float(s_clean.max())
            s_rescaled = s_clean.clip(lower=theta_r_lit)
            df["theta"] = s_rescaled.values

            n_above_ts = int((s_rescaled > theta_s_lit).sum())
            print(f"[DataClean-FIX-A] Dev{did}: range [{obs_min:.4f},{obs_max:.4f}] "
                  f"preserved. {n_above_ts} readings above θₛ_lit={theta_s_lit:.3f}")

            ts_from_data = float(np.percentile(s_clean.values, 99.0))
            ts_init      = max(ts_from_data, theta_r_lit + 0.05)
            ts_init      = min(ts_init, PTF_BOUNDS[did]["theta_s"][1])
            if did == 107:
                self.ts107_raw_init = ts_init
            else:
                self.ts108_raw_init = ts_init
            print(f"[FIX-C] Dev{did}: θₛ_init from 99th pct = {ts_init:.4f}")

            r = pd.to_numeric(df["rain"], errors="coerce").fillna(0.0).clip(lower=0)
            df["q_ms"] = r.values / 3.6e6

        # Verify val window contains rainfall
        val_mask_raw = ((df108_raw["t_h"] >= val_lo_h) &
                        (df108_raw["t_h"] < val_hi_h))
        rain_in_val = float(df108_raw.loc[val_mask_raw, "q_ms"].sum() * 3.6e6)
        print(f"[CHG-1] Rainfall in val window (Dev108): {rain_in_val:.1f} mm"
              f"  {'✓ OK' if rain_in_val > 1.0 else '⚠ WARNING: val window dry'}")

        # Verify test window contains rainfall
        te_mask_raw = (df108_raw["t_h"] >= te_lo_h)
        rain_in_te  = float(df108_raw.loc[te_mask_raw, "q_ms"].sum() * 3.6e6)
        print(f"[CHG-1] Rainfall in test window (Dev108): {rain_in_te:.1f} mm"
              f"  {'✓ OK' if rain_in_te > 1.0 else '⚠ WARNING: test window dry'}")

        # FIX 6: rain function from raw record before downsampling
        t108_raw_norm = (df108_raw["t_h"].values / T_MAX_TRAIN).astype(np.float64)
        q108_raw      = df108_raw["q_ms"].values.astype(np.float64)
        t107_raw_norm = (df107_raw["t_h"].values / T_MAX_TRAIN).astype(np.float64)
        q107_raw      = df107_raw["q_ms"].values.astype(np.float64)

        set_rain_fn(make_causal_rain_fn(
            t108_raw_norm, q108_raw,
            t107_raw_norm, q107_raw,
            float(X_NORM_108), float(X_NORM_107)
        ))
        print("[FIX 1/6] Causal rain function set from raw record.")

        # Valid PDE intervals (training window only — [0, T_MAX_TRAIN])
        def _get_segs(t_arr, gap_thresh):
            segs = []
            s = t_arr[0]
            for i in range(1, len(t_arr)):
                if t_arr[i] - t_arr[i-1] > gap_thresh:
                    segs.append((float(s), float(t_arr[i-1]))); s = t_arr[i]
            segs.append((float(s), float(t_arr[-1])))
            return segs

        segs108 = _get_segs(df108_raw["t_h"].values, GAP_THRESH_H)
        segs107 = _get_segs(df107_raw["t_h"].values, GAP_THRESH_H)
        all_segs = sorted(segs108 + segs107, key=lambda x: x[0])
        merged = [list(all_segs[0])]
        for s, e in all_segs[1:]:
            if s - merged[-1][1] <= GAP_THRESH_H:
                merged[-1][1] = max(merged[-1][1], e)
            else:
                merged.append([s, e])

        # Only keep intervals within training window
        valid_intervals_norm = []
        for s, e in merged:
            sn = s / T_MAX_TRAIN
            en = min(e, T_MAX_TRAIN) / T_MAX_TRAIN
            if en > sn + 1e-4 and sn < 1.0:
                valid_intervals_norm.append((float(sn), min(float(en), 1.0)))

        lengths = np.array([e - s for s, e in valid_intervals_norm])
        weights = lengths / lengths.sum()
        self.valid_intervals_norm = valid_intervals_norm
        self.interval_weights     = weights

        total_valid = sum((e - s) for s, e in valid_intervals_norm) * T_MAX_TRAIN
        print(f"[PDE] Valid collocation intervals: {len(valid_intervals_norm)} segs  "
              f"{total_valid:.1f}h / {T_MAX_TRAIN:.1f}h train window "
              f"({100*total_valid/T_MAX_TRAIN:.1f}% covered)")

        # CHG-3: DRY_STEP=10 for both devices
        df108 = _adaptive_ds(df108_raw, rain_col="q_ms", dry_step=DRY_STEP_108)
        df107 = _adaptive_ds(df107_raw, rain_col="q_ms", dry_step=DRY_STEP_107)

        z107 = SITE[107]["sensor_depth_m"] / Z_MAX
        z108 = SITE[108]["sensor_depth_m"] / Z_MAX
        df107["z_norm"] = z107; df107["x_norm"] = X_NORM_107
        df108["z_norm"] = z108; df108["x_norm"] = X_NORM_108
        df107["t_norm"] = df107["t_h"] / T_MAX_TRAIN
        df108["t_norm"] = df108["t_h"] / T_MAX_TRAIN

        self.df107 = df107; self.df108 = df108
        self.df107_raw = df107_raw; self.df108_raw = df108_raw
        self.z_norm_107 = z107; self.z_norm_108 = z108
        self.t_rain_h   = df108_raw["t_h"].values
        self.q_rain_ms  = df108_raw["q_ms"].values

        q_rain_max = float(max(df108_raw["q_ms"].max(), df107_raw["q_ms"].max(), 1e-12))
        self.q_rain_max = q_rain_max

        def _interp_rain(df_):
            q = np.interp(df_["t_h"].values,
                          df108_raw["t_h"].values,
                          df108_raw["q_ms"].values,
                          left=0.0, right=float(df108_raw["q_ms"].values[-1]))
            return torch.tensor(q / q_rain_max, dtype=torch.float32,
                                device=device).unsqueeze(1)

        def _tensors(df):
            def _t(c): return torch.tensor(df[c].values, dtype=torch.float32,
                                           device=device).unsqueeze(1)
            return _t("x_norm"), _t("z_norm"), _t("t_norm"), _t("theta")

        self.x108, self.z108, self.t108, self.th108 = _tensors(df108)
        self.x107, self.z107, self.t107, self.th107 = _tensors(df107)
        self.q108 = _interp_rain(df108)
        self.q107 = _interp_rain(df107)

        print(f"\n[Data] Dev107 (Sandy Clay, x=25m): {len(df107)} pts"
              f"  θ=[{df107['theta'].min():.3f},{df107['theta'].max():.3f}]")
        print(f"[Data] Dev108 (SCL, x=7m):         {len(df108)} pts"
              f"  θ=[{df108['theta'].min():.3f},{df108['theta'].max():.3f}]")
        print(f"[Data] T_MAX_SYNC={T_MAX_SYNC:.0f}h  T_MAX_FULL={T_MAX_FULL:.0f}h"
              f"  T_MAX_TRAIN={T_MAX_TRAIN:.1f}h")
        return self


# ═══════════════════════════════════════════════════════════════════════════
#  COLLOCATION SAMPLING
# ═══════════════════════════════════════════════════════════════════════════

def sample_collocation(n_int, n_bc_top, n_bc_bot, n_ic,
                       valid_intervals=None, interval_weights=None):
    def rn(n): return torch.rand(n, 1, device=device, requires_grad=True)
    def ze(n): return torch.zeros(n, 1, device=device, requires_grad=True)
    def on(n): return torch.ones(n, 1, device=device, requires_grad=True)

    def _sample_t_valid(n):
        if valid_intervals is None or len(valid_intervals) == 0:
            return torch.rand(n, 1, device=device)
        seg_idx = np.random.choice(len(valid_intervals), size=n, p=interval_weights)
        t_vals  = np.zeros(n, dtype=np.float32)
        for i, si in enumerate(seg_idx):
            s, e = valid_intervals[si]
            t_vals[i] = s + np.random.rand() * (e - s)
        return torch.tensor(t_vals, dtype=torch.float32, device=device).unsqueeze(1)

    n_each = n_int // 2
    def _jitter_x(val, n):
        base = torch.full((n, 1), val, device=device)
        return (base + 0.05*(torch.rand(n, 1, device=device)-0.5)).clamp(0., 1.)

    x_i = torch.cat([_jitter_x(X_NORM_107, n_each),
                     _jitter_x(X_NORM_108, n_int-n_each)], dim=0).requires_grad_(True)
    z_i = rn(n_int)
    t_i = _sample_t_valid(n_int).requires_grad_(True)

    x_bt = rn(n_bc_top);  z_bt = ze(n_bc_top)
    t_bt = _sample_t_valid(n_bc_top).requires_grad_(True)
    x_bb = rn(n_bc_bot);  z_bb = on(n_bc_bot)
    t_bb = _sample_t_valid(n_bc_bot).requires_grad_(True)
    x_ic = rn(n_ic);      z_ic = rn(n_ic);     t_ic = ze(n_ic)

    return (x_i,z_i,t_i), (x_bt,z_bt,t_bt), (x_bb,z_bb,t_bb), (x_ic,z_ic,t_ic)


# ═══════════════════════════════════════════════════════════════════════════
#  METRICS  (CHG-4: FoS statistics added)
# ═══════════════════════════════════════════════════════════════════════════

def _r2(o, p):
    ss = np.sum((o - o.mean())**2)
    return float(1 - np.sum((p - o)**2) / max(ss, 1e-9))

def _rmse(o, p):
    return float(np.sqrt(np.mean((p - o)**2)))

def compute_metrics_combined(model, data_loader):
    """
    CHG-4: Returns θ metrics AND FoS statistics per split and device.
    FoS is the primary output; θ metrics are used for training guidance.
    """
    results  = {}
    all_obs  = {s: [] for s in ["tr","val","te"]}
    all_pred = {s: [] for s in ["tr","val","te"]}
    all_fos  = {s: [] for s in ["tr","val","te"]}

    for did, x_t, z_t, t_t, th_t, q_t, df_ in [
        (108, data_loader.x108, data_loader.z108,
              data_loader.t108, data_loader.th108, data_loader.q108, data_loader.df108),
        (107, data_loader.x107, data_loader.z107,
              data_loader.t107, data_loader.th107, data_loader.q107, data_loader.df107),
    ]:
        t_h = df_["t_h"].values
        tr_m, val_m, te_m = _splits(t_h)

        with torch.no_grad():
            _, theta_pred, fos_pred = model(x_t, z_t, t_t)

        obs      = th_t.cpu().numpy().flatten()
        pred     = theta_pred.cpu().numpy().flatten()
        fos_np   = fos_pred.cpu().numpy().flatten()

        for mask, sname in [(tr_m,"tr"),(val_m,"val"),(te_m,"te")]:
            if mask.sum() > 2:
                results[f"r2_{sname}_{did}"]    = _r2(obs[mask],  pred[mask])
                results[f"rmse_{sname}_{did}"]  = _rmse(obs[mask], pred[mask])
                results[f"n_{sname}_{did}"]     = int(mask.sum())
                # CHG-4: FoS statistics
                results[f"fos_mean_{sname}_{did}"] = float(fos_np[mask].mean())
                results[f"fos_min_{sname}_{did}"]  = float(fos_np[mask].min())
                results[f"fos_max_{sname}_{did}"]  = float(fos_np[mask].max())
                results[f"fos_warn_{sname}_{did}"] = int((fos_np[mask] < 1.5).sum())
                results[f"fos_fail_{sname}_{did}"] = int((fos_np[mask] < 1.0).sum())
                all_obs[sname].append(obs[mask])
                all_pred[sname].append(pred[mask])
                all_fos[sname].append(fos_np[mask])

    for sname in ["tr","val","te"]:
        if all_obs[sname]:
            oc  = np.concatenate(all_obs[sname])
            pc  = np.concatenate(all_pred[sname])
            fc  = np.concatenate(all_fos[sname])
            results[f"r2_{sname}"]       = _r2(oc, pc)
            results[f"rmse_{sname}"]     = _rmse(oc, pc)
            results[f"n_{sname}"]        = len(oc)
            results[f"fos_mean_{sname}"] = float(fc.mean())
            results[f"fos_min_{sname}"]  = float(fc.min())
            results[f"fos_warn_{sname}"] = int((fc < 1.5).sum())
            results[f"fos_fail_{sname}"] = int((fc < 1.0).sum())

    return results


# ═══════════════════════════════════════════════════════════════════════════
#  TRAINING — STAGE 1  (CHG-2: single stage, no rolling window)
# ═══════════════════════════════════════════════════════════════════════════

PDE_NAN_WARN_THRESHOLD = 0.10

def train_stage1(data_loader,
                 n_seeds=3, total_epochs=15000, lr=2e-4,
                 lam_data=100.0,
                 lam_richards=20.0,
                 lam_bc_top=0.1,
                 lam_bc_bot=0.1,
                 lam_ic=0.0,
                 lam_smooth=0.01,
                 lam_prior=0.05,
                 lam_psi_var=1.0,
                 lam_fos_physics=0.1,       # CHG-4/5: FoS regularisation weight
                 es_patience=10000, anneal_epochs=4000,
                 n_int=2000, n_bc=400, n_ic=400):

    best_model  = None
    best_hist   = None
    all_results = []

    R2_MIN   = 0.55
    RMSE_MAX = 0.03

    ts107_init = data_loader.ts107_raw_init
    ts108_init = data_loader.ts108_raw_init

    for seed in range(n_seeds):
        set_seed(seed + 100)
        model = PINNSlope(ts107_init=ts107_init, ts108_init=ts108_init).to(device)
        opt   = torch.optim.Adam(model.parameters(), lr=lr)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    opt, mode='min', factor=0.5, patience=1500, min_lr=5e-6)

        hist           = defaultdict(list)
        best_val_score = -np.inf
        best_val_rmse  = np.inf
        wait           = 0
        best_state     = None
        nan_count      = 0

        print(f"\n{'─'*65}")
        print(f"[Stage1] Seed={seed}  lr={lr}  lam_data={lam_data}"
              f"  lam_pde={lam_richards}  lam_fos={lam_fos_physics}"
              f"  epochs={total_epochs}")
        print(f"  Split: {int(VAL_LO*100)}/{int((VAL_HI-VAL_LO)*100)}"
              f"/{int((1-TE_LO)*100)} (train/val/test)  "
              f"T_MAX_TRAIN={T_MAX_TRAIN:.1f}h")
        print(f"{'─'*65}")

        for ep in range(1, total_epochs+1):
            model.train()
            reset_nan_tracker()

            (x_i,z_i,t_i),(x_bt,z_bt,t_bt),(x_bb,z_bb,t_bb),(x_ic,z_ic,t_ic) = \
                sample_collocation(n_int, n_bc, n_bc, n_ic,
                                   valid_intervals=data_loader.valid_intervals_norm,
                                   interval_weights=data_loader.interval_weights)

            opt.zero_grad()
            anneal = min(1.0, ep / anneal_epochs)

            w108 = 1.0 + 24.0 * (data_loader.q108 > 0.001).float()
            w107 = 1.0 + 24.0 * (data_loader.q107 > 0.001).float()

            L_data = (lam_data * loss_data(model, data_loader.x108,
                          data_loader.z108, data_loader.t108, data_loader.th108, event_weights=w108) +
                      lam_data * loss_data(model, data_loader.x107,
                          data_loader.z107, data_loader.t107, data_loader.th107, event_weights=w107))
            L_pde  = anneal * lam_richards     * loss_richards(model, x_i, z_i, t_i)
            L_bct  = anneal * lam_bc_top        * loss_bc_top(model, x_bt, z_bt, t_bt)
            L_bcb  = anneal * lam_bc_bot        * loss_bc_bot(model, x_bb, z_bb, t_bb)
            L_ic   = (anneal * lam_ic           * loss_ic(model, x_ic, z_ic, t_ic)
                      if lam_ic > 0.0 else torch.tensor(0.0, device=device))
            L_sm   = anneal * lam_smooth        * loss_smoothness(model, x_i, z_i, t_i)
            L_pr   =          lam_prior         * loss_prior_vg(model)
            L_pvar = anneal * lam_psi_var       * loss_psi_variance(model, x_i, z_i, t_i)
            # CHG-4/5: FoS physical regularisation
            L_fos  = anneal * lam_fos_physics   * loss_fos_physics(model, x_i, z_i, t_i)

            L_total = L_data + L_pde + L_bct + L_bcb + L_ic + L_sm + L_pr + L_pvar + L_fos

            if not torch.isfinite(L_total):
                nan_count += 1; opt.zero_grad(); continue

            L_total.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            grads_ok = all(torch.isfinite(p.grad).all()
                           for p in model.parameters() if p.grad is not None)
            if grads_ok:
                opt.step()
            else:
                opt.zero_grad(); nan_count += 1

            hist["total"].append(L_total.item())
            hist["data"].append(L_data.item())
            hist["pde"].append(L_pde.item())
            hist["fos"].append(L_fos.item())

            if ep % 500 == 0 or ep == 1:
                model.eval()
                nan_fracs    = get_nan_fractions()
                pde_nan_frac = nan_fracs.get("richards", 0.0)
                pde_status   = ("OK" if pde_nan_frac <= PDE_NAN_WARN_THRESHOLD
                                else f"⚠ NaN={pde_nan_frac:.1%}")

                m      = compute_metrics_combined(model, data_loader)
                r2_val = m.get("r2_val",  float("nan"))
                rm_val = m.get("rmse_val",float("nan"))
                r2_tr  = m.get("r2_tr",   float("nan"))
                rm_tr  = m.get("rmse_tr", float("nan"))

                # CHG-4: report FoS statistics in training log
                fos_min_tr  = m.get("fos_min_tr",  float("nan"))
                fos_min_val = m.get("fos_min_val", float("nan"))
                fos_warn_val= m.get("fos_warn_val", 0)

                hist["r2_val"].append((ep, r2_val))
                hist["r2_tr"].append((ep, r2_tr))
                hist["pde_nan_frac"].append((ep, pde_nan_frac))

                cur_lr     = opt.param_groups[0]["lr"]
                r2_tr_108  = m.get("r2_tr_108",   float("nan"))
                r2_tr_107  = m.get("r2_tr_107",   float("nan"))
                rm_tr_108  = m.get("rmse_tr_108",  float("nan"))
                rm_tr_107  = m.get("rmse_tr_107",  float("nan"))
                r2_val_108 = m.get("r2_val_108",  float("nan"))
                r2_val_107 = m.get("r2_val_107",  float("nan"))
                rm_val_108 = m.get("rmse_val_108", float("nan"))
                rm_val_107 = m.get("rmse_val_107", float("nan"))

                meets_threshold = all(np.isfinite(v) for v in [
                    r2_tr_108, r2_tr_107, rm_tr_108, rm_tr_107,
                    r2_val_108, r2_val_107, rm_val_108, rm_val_107,
                ]) and (
                    r2_tr_108  >= R2_MIN and rm_tr_108  <= RMSE_MAX and
                    r2_tr_107  >= R2_MIN and rm_tr_107  <= RMSE_MAX and
                    r2_val_108 >= R2_MIN and rm_val_108 <= RMSE_MAX and
                    r2_val_107 >= R2_MIN and rm_val_107 <= RMSE_MAX
                )

                if meets_threshold:
                    r2_mean   = np.mean([r2_tr_108, r2_tr_107, r2_val_108, r2_val_107])
                    rmse_mean = np.mean([rm_tr_108, rm_tr_107, rm_val_108, rm_val_107])
                    val_score = r2_mean - 10.0 * rmse_mean
                else:
                    val_score = -np.inf

                if np.isfinite(rm_val) and rm_val < best_val_rmse:
                    best_val_rmse = rm_val
                    best_state    = {k: v.clone() for k, v in model.state_dict().items()}
                    es_tag = "  ✓ ckpt"
                else:
                    es_tag = ""

                if ep % DRIVE_CKPT_INTERVAL == 0:
                    _ckpt_path = os.path.join(DRIVE_CKPT_DIR,
                                              f"stage1_seed{seed}_ep{ep:05d}.pth")
                    torch.save({
                        "epoch": ep, "seed": seed,
                        "state_dict": {k: v.clone().cpu()
                                       for k, v in model.state_dict().items()},
                        "r2_val": r2_val, "rmse_val": rm_val,
                        "T_MAX_TRAIN": T_MAX_TRAIN,
                    }, _ckpt_path)
                    print(f"  [Drive] Interval ckpt → {_ckpt_path}")

                if val_score > best_val_score:
                    best_val_score = val_score; wait = 0
                    es_tag += "  ✓ best"
                else:
                    wait += 1
                    es_tag += f"  ES {wait*500}/{es_patience}"

                if np.isfinite(rm_val):
                    sched.step(rm_val)

                print(
                    f"  ep={ep:5d}/{total_epochs}"
                    f"  L={L_total.item():8.4f}"
                    f"  [data={L_data.item():.3f}"
                    f"  pde={L_pde.item():.4f}"
                    f"  fos={L_fos.item():.4f}"
                    f"  pde_nan={pde_nan_frac:.1%}]  [{pde_status}]"
                    f"  R²_tr={r2_tr:+.4f}  R²_val={r2_val:+.4f}"
                    f"  RMSE_val={rm_val:.5f}  lr={cur_lr:.1e}"
                    f"{es_tag}"
                )
                print(
                    f"         Dev108: R²_tr={r2_tr_108:+.4f}  R²_val={r2_val_108:+.4f}"
                    f"  |  Dev107: R²_tr={r2_tr_107:+.4f}  R²_val={r2_val_107:+.4f}"
                )
                # CHG-4: FoS line in training log
                print(
                    f"         FoS_min_tr={fos_min_tr:.3f}"
                    f"  FoS_min_val={fos_min_val:.3f}"
                    f"  FoS_warn_val={fos_warn_val}"
                )

                if wait * 500 >= es_patience:
                    print(f"\n  [EarlyStop] Seed={seed}  ep={ep}"
                          f"  best_val_RMSE={best_val_rmse:.5f}")
                    break
                model.train()

        if best_state:
            model.load_state_dict(best_state)
        model.eval()
        m = compute_metrics_combined(model, data_loader)

        def _dev_passes(did):
            for sname in ["tr","val"]:
                r2   = m.get(f"r2_{sname}_{did}",   float("nan"))
                rmse = m.get(f"rmse_{sname}_{did}", float("nan"))
                if not (np.isfinite(r2) and r2 >= R2_MIN and
                        np.isfinite(rmse) and rmse <= RMSE_MAX):
                    return False
            return True

        passes = _dev_passes(108) and _dev_passes(107)
        r2_tr  = m.get("r2_tr",   float("nan"))
        rm_tr  = m.get("rmse_tr", float("nan"))
        r2_val = m.get("r2_val",  float("nan"))
        rm_val = m.get("rmse_val",float("nan"))
        r2_te  = m.get("r2_te",   float("nan"))
        rm_te  = m.get("rmse_te", float("nan"))
        # CHG-4: FoS on test (primary output)
        fos_min_te   = m.get("fos_min_te",   float("nan"))
        fos_warn_te  = m.get("fos_warn_te",  0)
        fos_fail_te  = m.get("fos_fail_te",  0)

        status = "✓ PASS" if passes else "✗ FAIL"
        print(f"\n  [Seed {seed} Final] {status}"
              f"  Train R²={r2_tr:.4f} RMSE={rm_tr:.5f}"
              f"  Val R²={r2_val:.4f} RMSE={rm_val:.5f}"
              f"  Test R²={r2_te:.4f} RMSE={rm_te:.5f}"
              f"  nan_skips={nan_count}")
        print(f"  [FoS TEST] min={fos_min_te:.3f}  warn={fos_warn_te}  fail={fos_fail_te}")

        saved_state = {k: v.clone().cpu() for k, v in model.state_dict().items()}
        all_results.append(dict(
            seed=seed, state_dict=saved_state, hist=hist,
            r2_tr=r2_tr,   rmse_tr=rm_tr,
            r2_val=r2_val, rmse_val=rm_val,
            r2_te=r2_te,   rmse_te=rm_te,
            r2_tr_108=m.get("r2_tr_108",float("nan")),
            r2_tr_107=m.get("r2_tr_107",float("nan")),
            r2_val_108=m.get("r2_val_108",float("nan")),
            r2_val_107=m.get("r2_val_107",float("nan")),
            rmse_tr_108=m.get("rmse_tr_108",float("nan")),
            rmse_tr_107=m.get("rmse_tr_107",float("nan")),
            rmse_val_108=m.get("rmse_val_108",float("nan")),
            rmse_val_107=m.get("rmse_val_107",float("nan")),
            fos_min_te=fos_min_te,
            fos_warn_te=fos_warn_te,
            fos_fail_te=fos_fail_te,
            passes=passes,
            best_state_saved=(best_state is not None),
        ))

    def _absolute_score(r):
        r2_vals = [r.get(f"r2_{s}_{d}", float("nan"))
                   for s in ["tr","val"] for d in [108,107]]
        rm_vals = [r.get(f"rmse_{s}_{d}", float("nan"))
                   for s in ["tr","val"] for d in [108,107]]
        if not all(np.isfinite(v) for v in r2_vals + rm_vals):
            return -np.inf
        r2_mean   = float(np.mean(r2_vals))
        rmse_mean = float(np.mean(rm_vals))
        r2_score   = np.clip((r2_mean   - R2_MIN)  / 0.40,      0.0, 1.0)
        rmse_score = np.clip(1.0 - rmse_mean / RMSE_MAX,         0.0, 1.0)
        return 0.5 * r2_score + 0.5 * rmse_score

    def _passes_trainval(r):
        for did in [108, 107]:
            for sname in ["tr", "val"]:
                r2   = r.get(f"r2_{sname}_{did}",   float("nan"))
                rmse = r.get(f"rmse_{sname}_{did}", float("nan"))
                if not (np.isfinite(r2) and r2 >= R2_MIN and
                        np.isfinite(rmse) and rmse <= RMSE_MAX):
                    return False
        return True

    def _positive_trainval_both(r):
        return all(r.get(f"r2_{s}_{d}", float("nan")) > 0.0
                   for s in ["tr","val"] for d in [108,107])

    finite = [r for r in all_results
              if np.isfinite(r["r2_val"]) and np.isfinite(r["rmse_val"])
              and r.get("best_state_saved", True)]

    if not finite:
        raise RuntimeError("All seeds produced NaN metrics or no checkpoint was saved.")

    tier1 = [r for r in finite if _passes_trainval(r)]
    tier2 = [r for r in finite if _positive_trainval_both(r)]
    tier3 = [r for r in finite if r.get("r2_val", float("nan")) > 0.0]
    tier4 = finite

    if tier1:    pool = tier1; tier_name = "Tier 1 (threshold pass)"
    elif tier2:  pool = tier2; tier_name = "Tier 2 (positive R² both devices) — retraining advised"
    elif tier3:  pool = tier3; tier_name = "Tier 3 (positive combined val R²) — retraining advised"
    else:        pool = tier4; tier_name = "Tier 4 (fallback: all seeds failed)"

    best_result = max(pool, key=_absolute_score)
    best_model  = PINNSlope(ts107_init=ts107_init, ts108_init=ts108_init).to(device)
    best_model.load_state_dict(
        {k: v.to(device) for k, v in best_result["state_dict"].items()})
    best_model.eval()
    best_hist = best_result["hist"]

    print(f"\n[Stage1 Best] {tier_name}")
    print(f"  Seed={best_result['seed']}"
          f"  Train R²={best_result['r2_tr']:.4f}  RMSE={best_result['rmse_tr']:.5f}")
    print(f"  Val   R²={best_result['r2_val']:.4f}  RMSE={best_result['rmse_val']:.5f}")
    print(f"  Test  R²={best_result['r2_te']:.4f}  RMSE={best_result['rmse_te']:.5f}"
          f"  ← strictly unseen")
    print(f"  Per-device: "
          f"Dev108 R²_tr={best_result.get('r2_tr_108',float('nan')):+.4f} "
          f"R²_val={best_result.get('r2_val_108',float('nan')):+.4f}  | "
          f"Dev107 R²_tr={best_result.get('r2_tr_107',float('nan')):+.4f} "
          f"R²_val={best_result.get('r2_val_107',float('nan')):+.4f}")
    # CHG-4: FoS primary output summary
    print(f"  [FoS PRIMARY OUTPUT] Test: min={best_result['fos_min_te']:.3f}"
          f"  warnings={best_result['fos_warn_te']}"
          f"  failures={best_result['fos_fail_te']}")
    if tier_name.startswith("Tier 1"):
        print(f"  [OK] Model meets publishable thresholds "
              f"(R²>={R2_MIN}, RMSE<={RMSE_MAX} on all train/val splits)")
    else:
        print(f"  [WARNING] Model does NOT meet publishable thresholds")

    return best_model, best_hist, all_results


# ═══════════════════════════════════════════════════════════════════════════
#  FIGURE HELPERS
# ═══════════════════════════════════════════════════════════════════════════

matplotlib.rcParams.update({
    "font.family":"DejaVu Sans","font.size":11,
    "axes.titlesize":12,"axes.labelsize":11,
    "xtick.labelsize":10,"ytick.labelsize":10,
    "legend.fontsize":9,"figure.dpi":150,
})

_OUT = "iot_figures"
os.makedirs(_OUT, exist_ok=True)

def _save(fig, name):
    p = os.path.join(_OUT, name)
    fig.savefig(p, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"  [Fig] {name}"); plt.close(fig)

def _pred_ts(model, df, x_norm_val, z_norm_val, batch=4096):
    t_np = df["t_h"].values.astype(np.float32)
    pred = np.zeros(len(t_np), dtype=np.float32)
    with torch.no_grad():
        for i in range(0, len(t_np), batch):
            t_b = torch.tensor(t_np[i:i+batch]/T_MAX_TRAIN, dtype=torch.float32,
                               device=device).unsqueeze(1)
            x_b = torch.full_like(t_b, x_norm_val)
            z_b = torch.full_like(t_b, z_norm_val)


            _, th, _ = model(x_b, z_b, t_b)
            pred[i:i+batch] = th.cpu().numpy().flatten()
    return t_np, pred

def _pred_fos_ts(model, x_norm_val, t_h_arr):
    t_np = t_h_arr.astype(np.float32)
    fos  = np.zeros(len(t_np), dtype=np.float32)
    with torch.no_grad():
        for i in range(0, len(t_np), 2048):
            t_b = torch.tensor(t_np[i:i+2048]/T_MAX_TRAIN, dtype=torch.float32,
                               device=device).unsqueeze(1)
            x_b = torch.full_like(t_b, x_norm_val)
            z_b = torch.ones_like(t_b)
            _, _, fp = model(x_b, z_b, t_b)
            fos[i:i+2048] = fp.cpu().numpy().flatten()
    return np.clip(fos, 0.1, 15.0)

def _persistence_pred(obs, t_h, window_h=24):
    pred = np.zeros_like(obs)
    dt   = np.median(np.diff(t_h)) if len(t_h) > 1 else 1.0
    lag  = max(1, int(window_h / dt))
    pred[lag:] = obs[:-lag]; pred[:lag] = obs[0]
    return pred

def _persistence_24h_rmse(obs_np, t_h_np):
    if len(obs_np) < 2:
        return float(np.std(obs_np)) + 1e-6
    dt_h    = np.median(np.diff(t_h_np))
    if dt_h <= 0: dt_h = 1.0
    lag_idx = max(1, int(round(24.0 / dt_h)))
    if lag_idx >= len(obs_np): lag_idx = len(obs_np) - 1
    return float(np.sqrt(np.mean((obs_np[lag_idx:] - obs_np[:-lag_idx])**2)))

def _lstm_pred(obs_all, seq_len=24):
    try:
        n_tr   = int(len(obs_all)*0.70)
        obs_tr = obs_all[:n_tr]
        X, Y   = [], []
        for i in range(seq_len, n_tr):
            X.append(obs_tr[max(0,i-seq_len):i]); Y.append(obs_tr[i])
        if len(X) < 10: return np.full_like(obs_all, obs_all.mean())
        X    = torch.tensor(np.array(X), dtype=torch.float32).unsqueeze(-1)
        Y    = torch.tensor(np.array(Y), dtype=torch.float32).unsqueeze(-1)
        lstm = nn.LSTM(1, 32, batch_first=True); head = nn.Linear(32, 1)
        opt  = torch.optim.Adam(list(lstm.parameters())+list(head.parameters()), lr=1e-3)
        for _ in range(200):
            opt.zero_grad()
            out, _ = lstm(X); ((head(out[:,-1,:]) - Y)**2).mean().backward(); opt.step()
        pred = np.zeros(len(obs_all)); pred[:seq_len] = obs_all[:seq_len]
        lstm.eval()
        with torch.no_grad():
            for i in range(seq_len, len(obs_all)):
                xin   = torch.tensor(pred[i-seq_len:i], dtype=torch.float32).unsqueeze(0).unsqueeze(-1)
                out,_ = lstm(xin); pred[i] = head(out[0,-1,:]).item()
        return pred
    except Exception:
        return np.full_like(obs_all, obs_all.mean())


# ═══════════════════════════════════════════════════════════════════════════
#  FIGURES
# ═══════════════════════════════════════════════════════════════════════════

def _fig1_convergence(train_hist, all_seed_results):
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle("Stage 1 training convergence  [80/10/10 split]",
                 fontweight="bold", fontsize=13)
    colors = ["#2196F3", "#FF5722", "#4CAF50"]
    for si, res in enumerate(all_seed_results):
        h   = res["hist"]
        col = colors[si % len(colors)]
        lbl = f"Seed {res['seed']}"
        def _sc(r):
            r2s = [r.get(f"r2_{s}_{d}", float("nan")) for s in ["tr","val"] for d in [108,107]]
            rms = [r.get(f"rmse_{s}_{d}", float("nan")) for s in ["tr","val"] for d in [108,107]]
            if not all(np.isfinite(v) for v in r2s+rms): return -np.inf
            return float(np.mean(r2s)) - 10.0*float(np.mean(rms))
        star = " ★" if _sc(res) == max(_sc(r) for r in all_seed_results) else ""
        if h.get("total"):
            axes[0,0].semilogy(h["total"], color=col, alpha=0.75, lw=1.2, label=lbl+star)
        if h.get("data"):
            axes[0,1].semilogy(h["data"], color=col, alpha=0.75, lw=1.2, label=lbl+star)
        if h.get("pde"):
            axes[1,0].semilogy([max(v,1e-10) for v in h["pde"]], color=col, alpha=0.75,
                               lw=1.2, label=lbl+star)
        if h.get("r2_val"):
            ep_v, r2_v = zip(*h["r2_val"])
            axes[1,1].plot(ep_v, r2_v, color=col, alpha=0.85, lw=1.4,
                           marker="o", ms=3, label=lbl+star)
    for ax, ttl in zip(axes.flat,
                       ["Total loss", "Data loss", "PDE (Richards) loss", "Val R² history"]):
        ax.set_title(ttl, fontsize=11)
        ax.set_xlabel("Epoch"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
        if ttl != "Val R² history":
            ax.set_ylabel("Loss")
        else:
            ax.axhline(0.55, color="red", ls="--", lw=1.2, label="R²=0.55 gate")
            ax.set_ylabel("R²"); ax.set_ylim(-0.5, 1.05)
    plt.tight_layout()
    _save(fig, "fig1_convergence.png")


def _fig2_theta_timeseries(model, data_loader):
    df108 = data_loader.df108; df107 = data_loader.df107
    t_rain_h    = data_loader.t_rain_h
    q_rain_mmhr = data_loader.q_rain_ms * 3.6e6

    t108_h, pred108 = _pred_ts(model, df108, X_NORM_108, data_loader.z_norm_108)
    t107_h, pred107 = _pred_ts(model, df107, X_NORM_107, data_loader.z_norm_107)
    obs108 = df108["theta"].values; obs107 = df107["theta"].values
    tr108, val108, te108 = _splits(t108_h)
    tr107, val107, te107 = _splits(t107_h)

    fig = plt.figure(figsize=(16, 12))
    gs_ = _gs.GridSpec(4, 1, height_ratios=[0.8, 2.5, 2.5, 0.8], hspace=0.40)
    fig.suptitle(
        "Volumetric water content θ: observed vs PINN predicted\n"
        "Split: 80% train | 10% val | 10% test (unseen)",
        fontweight="bold", fontsize=12)

    ax0 = fig.add_subplot(gs_[0])
    ax0.bar(t_rain_h, q_rain_mmhr, width=2, color="#4A90D9", alpha=0.8)
    ax0.set_ylabel("Rain (mm/hr)"); ax0.set_xlim(0, T_MAX)
    ax0.set_title("Rainfall input", fontsize=10)
    ax0.axvline(T_MAX_SYNC, color="purple", ls=":", lw=1.5, label="Dev107 ends")
    # CHG-1: show val and test bands
    ax0.axvspan(T_MAX_SYNC * VAL_LO, T_MAX_SYNC * VAL_HI,
                alpha=0.15, color="#FF9800", label=f"Val [80–90%]")
    ax0.axvspan(T_MAX_SYNC * TE_LO,  T_MAX_SYNC * 1.00,
                alpha=0.12, color="#E91E63", label=f"Test [90–100%] unseen")
    ax0.legend(fontsize=8)

    for ax, obs, pred, t_h, tr_m, val_m, te_m, did, soil, depth in [
        (fig.add_subplot(gs_[1]), obs108, pred108, t108_h, tr108, val108, te108,
         108, "Sandy Clay Loam", "30 cm"),
        (fig.add_subplot(gs_[2]), obs107, pred107, t107_h, tr107, val107, te107,
         107, "Sandy Clay", "22 cm"),
    ]:
        ax.axvspan(T_MAX_SYNC * VAL_LO, T_MAX_SYNC * VAL_HI, alpha=0.10, color="#FF9800")
        ax.axvspan(T_MAX_SYNC * TE_LO,  T_MAX_SYNC * 1.00,  alpha=0.08, color="#E91E63")
        ax.plot(t_h, obs,  color="#333",    lw=0.9, alpha=0.9, label="Observed θ")
        ax.plot(t_h, pred, color="#E91E63", lw=1.3, alpha=0.85, ls="--",
                label="PINN predicted θ")
        for mask, sname, col in [(tr_m,"Train","#2196F3"),
                                  (val_m,"Val","#FF9800"),
                                  (te_m,"Test (unseen)","#E91E63")]:
            if mask.sum() > 2:
                r2  = _r2(obs[mask],  pred[mask])
                rm  = _rmse(obs[mask], pred[mask])
                mid = t_h[mask].mean()
                ypos = obs.max() + 0.01
                ax.text(mid, ypos,
                        f"{sname}\nR²={r2:.3f}\nRMSE={rm:.4f}",
                        ha="center", va="bottom", fontsize=7.5, color=col,
                        bbox=dict(fc="white", ec=col, alpha=0.7, pad=2))
        ax.axvline(T_MAX_SYNC, color="purple", ls=":", lw=1.5)
        ax.set_ylabel("θ (m³/m³)"); ax.set_xlim(0, T_MAX)
        ax.set_title(f"Dev{did} — {soil}, x={SITE[did]['x_pos']:.0f} m, z={depth}",
                     fontsize=10)
        ax.legend(fontsize=8, loc="upper right")
        ax.set_ylim(min(THETA_LO_107, THETA_LO_108) - 0.02,
                    max(THETA_HI_107, THETA_HI_108) + 0.04)

    ax3 = fig.add_subplot(gs_[3])
    res108 = pred108 - obs108
    res107 = pred107[:len(obs107)] - obs107
    ax3.plot(t108_h, res108, color="#D45F5F", lw=0.8, alpha=0.8, label="Residual Dev108")
    ax3.plot(t107_h, res107, color="#4A90D9", lw=0.8, alpha=0.7, label="Residual Dev107")
    ax3.axhline(0, color="k", lw=0.8, ls="--")
    ax3.axhline( 0.03, color="gray", ls=":", lw=0.8)
    ax3.axhline(-0.03, color="gray", ls=":", lw=0.8)
    ax3.axvspan(T_MAX_SYNC * VAL_LO, T_MAX_SYNC * VAL_HI, alpha=0.08, color="#FF9800")
    ax3.axvspan(T_MAX_SYNC * TE_LO,  T_MAX_SYNC * 1.00,  alpha=0.06, color="#E91E63")
    ax3.set_ylabel("Residual (pred−obs)"); ax3.set_xlabel("Time (h)")
    ax3.set_xlim(0, T_MAX); ax3.set_ylim(-0.08, 0.08)
    ax3.legend(fontsize=8)
    ax3.set_title("Prediction residuals  (orange=val | red=test unseen)", fontsize=10)
    _save(fig, "fig2_theta_timeseries.png")
    return (t108_h, pred108, obs108, tr108, val108, te108,
            t107_h, pred107, obs107, tr107, val107, te107)


def _fig3_scatter_residuals(t108_h, pred108, obs108, tr108, val108, te108,
                             t107_h, pred107, obs107, tr107, val107, te107):
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle("Predicted vs observed scatter and residual distributions  [80/10/10]",
                 fontweight="bold", fontsize=13)
    split_defs = [("Train (80%)", tr108, tr107, "#2196F3"),
                  ("Val (10%)",   val108, val107, "#FF9800"),
                  ("Test (10%) — unseen", te108, te107, "#E91E63")]
    for col_i, (sname, m108, m107, col) in enumerate(split_defs):
        oc = np.concatenate([obs108[m108], obs107[m107]])
        pc = np.concatenate([pred108[m108], pred107[:len(obs107)][m107]])
        if len(oc) == 0: continue
        r2  = _r2(oc, pc); rm = _rmse(oc, pc)
        mn  = min(oc.min(), pc.min()) - 0.005
        mx  = max(oc.max(), pc.max()) + 0.005
        ax  = axes[0, col_i]
        ax.scatter(oc, pc, s=6, alpha=0.45, color=col)
        ax.plot([mn, mx], [mn, mx], "k--", lw=1.2, label="1:1")
        ax.set_xlabel("Observed θ"); ax.set_ylabel("Predicted θ")
        ax.set_title(f"{sname}\nR²={r2:.4f}  RMSE={rm:.5f}  (n={len(oc)})", fontsize=10)
        ax.set_xlim(mn, mx); ax.set_ylim(mn, mx); ax.set_aspect("equal")
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
        ax2  = axes[1, col_i]
        res  = pc - oc
        ax2.hist(res, bins=40, color=col, alpha=0.75, edgecolor="white")
        ax2.axvline(0, color="k", lw=1.2, ls="--")
        ax2.axvline(res.mean(), color="red", lw=1.2,
                    label=f"mean={res.mean():.4f}")
        ax2.axvline(res.mean()+res.std(), color="gray", lw=0.9, ls=":",
                    label=f"±1σ={res.std():.4f}")
        ax2.axvline(res.mean()-res.std(), color="gray", lw=0.9, ls=":")
        ax2.set_xlabel("Residual (pred−obs)"); ax2.set_ylabel("Count")
        ax2.set_title(f"Residual distribution — {sname}", fontsize=10)
        ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    _save(fig, "fig3_scatter_residuals.png")


def _fig4_vg_curves(model):
    p         = model.get_learned_params()
    psi_range = np.linspace(-8, 0, 500)
    def vg_theta_np(psi, alpha, n, theta_r, theta_s):
        m   = 1.0 - 1.0/n
        arg = np.abs(psi) * alpha
        Se  = np.where(psi >= 0, 1.0, 1.0 / (1.0 + arg**n)**m)
        Se  = np.clip(Se, 1e-6, 1.0)
        return theta_r + (theta_s - theta_r) * Se
    fig, axes = plt.subplots(1, 2, figsize=(13, 6))
    fig.suptitle("Van Genuchten retention curves: PTF prior vs PINN-learned",
                 fontweight="bold", fontsize=13)
    for did, title, ax in [
        (107, "Dev107 Sandy Clay (x=25 m)", axes[0]),
        (108, "Dev108 Sandy Clay Loam (x=7 m)", axes[1]),
    ]:
        s        = SITE[did]
        th_ptf   = vg_theta_np(psi_range, s["alpha"], s["n_vg"], s["theta_r"], s["theta_s"])
        th_learn = vg_theta_np(psi_range, p[f"alpha_{did}"], p[f"n_vg_{did}"],
                               p[f"theta_r_{did}"], p[f"theta_s_{did}"])
        bounds   = PTF_BOUNDS[did]
        th_lo    = vg_theta_np(psi_range, bounds["alpha"][0], bounds["n_vg"][0],
                               bounds["theta_r"][0], bounds["theta_s"][0])
        th_hi    = vg_theta_np(psi_range, bounds["alpha"][1], bounds["n_vg"][1],
                               bounds["theta_r"][1], bounds["theta_s"][1])
        ax.fill_between(psi_range, th_lo, th_hi, alpha=0.15, color="#2196F3",
                        label="PTF ±20–30% bounds")
        ax.plot(psi_range, th_ptf,   "b--", lw=1.8, label="PTF prior (C&P 1988)")
        ax.plot(psi_range, th_learn, "r-",  lw=2.2, label="PINN-learned")
        ax.set_xlabel("Matric potential ψ (m)"); ax.set_ylabel("θ (m³/m³)")
        ax.set_title(title, fontsize=10); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
        ax.set_xlim(-8, 0.2)
        txt = (f"Learned: α={p[f'alpha_{did}']:.3f}  n={p[f'n_vg_{did}']:.4f}\n"
               f"         θ_r={p[f'theta_r_{did}']:.4f}  θ_s={p[f'theta_s_{did}']:.4f}\n"
               f"Prior:   α={s['alpha']:.3f}  n={s['n_vg']:.4f}\n"
               f"         θ_r={s['theta_r']:.4f}  θ_s={s['theta_s']:.4f}")
        ax.text(0.02, 0.98, txt, transform=ax.transAxes, fontsize=7.5, va="top",
                family="monospace",
                bbox=dict(fc="white", ec="gray", alpha=0.8, pad=3))
    plt.tight_layout()
    _save(fig, "fig4_vg_curves.png")


def _fig5_fos_primary(model, data_loader):
    """
    CHG-4: Fig 5 — Factor of Safety as PRIMARY output.

    Panel 1: Rainfall
    Panel 2: FoS time series for Dev107 and Dev108 — full record, split annotated
    Panel 3: FoS statistics per split (train/val/test) — box-style summary
    Panel 4: ψ time series (intermediate state underlying FoS)
    """
    df108 = data_loader.df108; df107 = data_loader.df107
    t_rain_h    = data_loader.t_rain_h
    q_rain_mmhr = data_loader.q_rain_ms * 3.6e6

    # FoS time series
    t_h_arr_108 = df108["t_h"].values
    t_h_arr_107 = df107["t_h"].values
    fos108 = _pred_fos_ts(model, X_NORM_108, t_h_arr_108)
    fos107 = _pred_fos_ts(model, X_NORM_107, t_h_arr_107)

    # ψ time series (from theta prediction chain)
    psi108 = np.zeros(len(t_h_arr_108))
    psi107 = np.zeros(len(t_h_arr_107))
    with torch.no_grad():
        for arr, xn, result in [
            (t_h_arr_108, X_NORM_108, psi108),
            (t_h_arr_107, X_NORM_107, psi107),
        ]:
            for i in range(0, len(arr), 2048):
                t_b = torch.tensor(arr[i:i+2048]/T_MAX_TRAIN, dtype=torch.float32,
                                   device=device).unsqueeze(1)
                x_b = torch.full_like(t_b, xn)
                z_b = torch.ones_like(t_b)
                psi_b, _, _ = model(x_b, z_b, t_b)
                result[i:i+2048] = psi_b.cpu().numpy().flatten()

    tr108, val108, te108 = _splits(t_h_arr_108)
    tr107, val107, te107 = _splits(t_h_arr_107)

    fig = plt.figure(figsize=(16, 16))
    gs_ = _gs.GridSpec(4, 1, height_ratios=[0.8, 3.0, 2.0, 2.0], hspace=0.42)
    fig.suptitle(
        "Factor of Safety (FoS) — PRIMARY MODEL OUTPUT\n"
        "Infinite slope formulation  [failure plane = 1.0 m]  "
        "Split: 80% train | 10% val | 10% test (unseen)",
        fontweight="bold", fontsize=12)

    # --- Panel 0: Rainfall ---
    ax0 = fig.add_subplot(gs_[0])
    ax0.bar(t_rain_h, q_rain_mmhr, width=2, color="#4A90D9", alpha=0.8)
    ax0.set_ylabel("Rainfall\n(mm/hr)"); ax0.set_xlim(0, T_MAX)
    ax0.axvline(T_MAX_SYNC, color="purple", ls=":", lw=1.5)
    ax0.axvspan(T_MAX_SYNC * VAL_LO, T_MAX_SYNC * VAL_HI, alpha=0.13, color="#FF9800",
                label="Val 10%")
    ax0.axvspan(T_MAX_SYNC * TE_LO,  T_MAX_SYNC,          alpha=0.10, color="#E91E63",
                label="Test 10% (unseen)")
    ax0.legend(fontsize=8); ax0.set_title("Rainfall input", fontsize=10)

    # --- Panel 1: FoS time series ---
    ax1 = fig.add_subplot(gs_[1])
    ax1.axvspan(T_MAX_SYNC * VAL_LO, T_MAX_SYNC * VAL_HI, alpha=0.10, color="#FF9800")
    ax1.axvspan(T_MAX_SYNC * TE_LO,  T_MAX_SYNC,          alpha=0.08, color="#E91E63")
    ax1.plot(t_h_arr_108, fos108, color="#D45F5F", lw=1.4, alpha=0.9,
             label="FoS Dev108 (x=7m, toe)")
    ax1.plot(t_h_arr_107, fos107, color="#4A90D9", lw=1.4, alpha=0.85,
             label="FoS Dev107 (x=25m, mid)")
    ax1.axhline(1.5, color="orange", ls="--", lw=2.0, label="Warning  FoS=1.5")
    ax1.axhline(1.0, color="red",    ls="--", lw=2.5, label="Failure  FoS=1.0")
    ax1.axvline(T_MAX_SYNC, color="purple", ls=":", lw=1.5, label="Dev107 end")
    # Annotate split boundaries
    for frac, lbl, col in [(VAL_LO,"Val→","#FF9800"),(TE_LO,"Test→","#E91E63")]:
        ax1.axvline(T_MAX_SYNC*frac, color=col, ls="-.", lw=1.5)
        ax1.text(T_MAX_SYNC*frac + 20, 13.5, lbl, color=col, fontsize=8,
                 fontweight="bold")
    ax1.set_ylabel("Factor of Safety (FoS)"); ax1.set_xlim(0, T_MAX)
    ax1.set_ylim(0.3, 15.0); ax1.legend(fontsize=8, ncol=2)
    ax1.set_title("FoS time series — full record (train + val + test)", fontsize=10)
    # Add FoS statistics text boxes per split
    for mask108, mask107, sname, col, xfrac in [
        (tr108, tr107,  "Train",       "#2196F3", VAL_LO*0.5),
        (val108, val107,"Val",          "#FF9800", (VAL_LO+VAL_HI)/2),
        (te108, te107,  "Test\n(unseen)","#E91E63", (TE_LO+1.0)/2),
    ]:
        fc = np.concatenate([fos108[mask108], fos107[mask107]])
        if len(fc) == 0: continue
        ax1.text(T_MAX_SYNC*xfrac, 1.2,
                 f"{sname}\nFoS: {fc.min():.2f}–{fc.max():.2f}\n"
                 f"warn={int((fc<1.5).sum())}  fail={int((fc<1.0).sum())}",
                 ha="center", fontsize=7.5, color=col,
                 bbox=dict(fc="white", ec=col, alpha=0.8, pad=2))

    # --- Panel 2: FoS by split (bar summary) ---
    ax2 = fig.add_subplot(gs_[2])
    split_colors = {"Train":"#2196F3", "Val":"#FF9800", "Test (unseen)":"#E91E63"}
    split_data   = {}
    for mask108, mask107, sname in [
        (tr108, tr107, "Train"), (val108, val107, "Val"), (te108, te107, "Test (unseen)")
    ]:
        fc = np.concatenate([fos108[mask108], fos107[mask107]])
        split_data[sname] = fc
    x_pos = np.arange(len(split_data))
    bw    = 0.35
    for i, (sname, fc) in enumerate(split_data.items()):
        if len(fc) == 0: continue
        col = split_colors[sname]
        ax2.bar(i, fc.mean(), width=bw, color=col, alpha=0.75, label=sname)
        ax2.errorbar(i, fc.mean(), yerr=[[fc.mean()-fc.min()],[fc.max()-fc.mean()]],
                     fmt="none", color="k", capsize=4, lw=1.5)
        ax2.text(i, fc.min() - 0.15, f"min={fc.min():.2f}", ha="center",
                 fontsize=8, color=col)
    ax2.axhline(1.5, color="orange", ls="--", lw=1.5, label="Warn 1.5")
    ax2.axhline(1.0, color="red",    ls="--", lw=1.8, label="Fail 1.0")
    ax2.set_xticks(x_pos); ax2.set_xticklabels(list(split_data.keys()))
    ax2.set_ylabel("Factor of Safety (FoS)")
    ax2.set_title("FoS summary per split (bar=mean, error=min–max)", fontsize=10)
    ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3, axis="y")

    # --- Panel 3: ψ time series ---
    ax3 = fig.add_subplot(gs_[3])
    ax3.axvspan(T_MAX_SYNC * VAL_LO, T_MAX_SYNC * VAL_HI, alpha=0.10, color="#FF9800")
    ax3.axvspan(T_MAX_SYNC * TE_LO,  T_MAX_SYNC,          alpha=0.08, color="#E91E63")
    ax3.plot(t_h_arr_108, psi108, color="#D45F5F", lw=1.2, label="ψ Dev108 (x=7m)")
    ax3.plot(t_h_arr_107, psi107, color="#4A90D9", lw=1.2, alpha=0.85,
             label="ψ Dev107 (x=25m)")
    ax3.axhline(0.0, color="k", ls="--", lw=1.0, label="ψ=0 (saturation)")
    ax3.set_ylabel("Matric suction ψ (m)"); ax3.set_xlabel("Time (h)")
    ax3.set_xlim(0, T_MAX)
    ax3.set_title("Matric suction ψ — intermediate state driving FoS", fontsize=10)
    ax3.legend(fontsize=8)
    _save(fig, "fig5_fos_primary.png")


def _fig6_event_zoom(model, data_loader):
    df108 = data_loader.df108; df107 = data_loader.df107
    t_rain_h    = data_loader.t_rain_h
    q_rain_mmhr = data_loader.q_rain_ms * 3.6e6
    rain_smooth = np.convolve(q_rain_mmhr, np.ones(24)/24, mode="same")
    peak_idxs = []
    used = set()
    for idx in np.argsort(rain_smooth)[::-1]:
        if all(abs(idx-j) > 200 for j in used):
            peak_idxs.append(idx); used.add(idx)
        if len(peak_idxs) == 3: break
    fig, axes = plt.subplots(3, 3, figsize=(16, 12))
    fig.suptitle("Event-level zoom: rainfall → θ → FoS response",
                 fontweight="bold", fontsize=13)
    for row, pidx in enumerate(peak_idxs):
        t_peak = t_rain_h[pidx]
        t_lo   = max(0, t_peak - 120)
        t_hi   = min(T_MAX_FULL, t_peak + 180)
        ax0 = axes[row, 0]
        mask_r = (t_rain_h >= t_lo) & (t_rain_h <= t_hi)
        ax0.bar(t_rain_h[mask_r], q_rain_mmhr[mask_r], width=1.5, color="#4A90D9", alpha=0.85)
        ax0.axvline(t_peak, color="red", lw=1.2, ls="--")
        ax0.set_title(f"Event {row+1}: rain  (peak t={t_peak:.0f}h)", fontsize=9)
        ax0.set_xlabel("Time (h)"); ax0.set_ylabel("mm/hr")
        ax1 = axes[row, 1]
        for df_, xn, zn, col, lbl in [
            (df108, X_NORM_108, data_loader.z_norm_108, "#D45F5F", "Dev108"),
            (df107, X_NORM_107, data_loader.z_norm_107, "#4A90D9", "Dev107"),
        ]:
            mask_d = (df_["t_h"].values >= t_lo) & (df_["t_h"].values <= t_hi)
            if mask_d.sum() < 2: continue
            df_win = df_[mask_d]
            t_h_win, pred_win = _pred_ts(model, df_win, xn, zn)
            ax1.plot(df_win["t_h"].values, df_win["theta"].values,
                     color=col, lw=1.0, alpha=0.8, label=f"{lbl} obs")
            ax1.plot(t_h_win, pred_win, color=col, lw=1.5, ls="--", label=f"{lbl} pred")
        ax1.axvline(t_peak, color="red", lw=1.2, ls="--")
        ax1.set_title(f"Event {row+1}: θ response", fontsize=9)
        ax1.set_xlabel("Time (h)"); ax1.set_ylabel("θ (m³/m³)")
        ax1.legend(fontsize=7); ax1.grid(True, alpha=0.3)
        ax2 = axes[row, 2]
        t_fine    = np.linspace(t_lo, t_hi, 400)
        fos108_w  = _pred_fos_ts(model, X_NORM_108, t_fine)
        fos107_w  = _pred_fos_ts(model, X_NORM_107, t_fine)
        ax2.plot(t_fine, fos108_w, color="#D45F5F", lw=1.5, label="FoS Dev108")
        ax2.plot(t_fine, fos107_w, color="#4A90D9", lw=1.5, label="FoS Dev107", alpha=0.85)
        ax2.axhline(1.5, color="orange", ls="--", lw=1.5, label="Warn 1.5")
        ax2.axhline(1.0, color="red",    ls="--", lw=2.0, label="Fail 1.0")
        ax2.axvline(t_peak, color="red", lw=1.2, ls="--")
        ax2.set_title(f"Event {row+1}: FoS (primary output)", fontsize=9)
        ax2.set_xlabel("Time (h)"); ax2.set_ylabel("FoS")
        ax2.set_ylim(0.5, 8.0); ax2.legend(fontsize=7); ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    _save(fig, "fig6_event_zoom.png")


def _fig7_spatial_fos(model, data_loader):
    t_rain_h    = data_loader.t_rain_h
    q_rain_mmhr = data_loader.q_rain_ms * 3.6e6
    rain_smooth = np.convolve(q_rain_mmhr, np.ones(12)/12, mode="same")
    dry_times   = t_rain_h[np.argsort(rain_smooth)[:3]]
    wet_times   = t_rain_h[np.argsort(rain_smooth)[-3:]]
    t_stamps    = np.sort(np.concatenate([
        dry_times[:2], wet_times[-2:],
        [T_MAX_SYNC*0.4, T_MAX_SYNC*0.7]
    ]))
    t_stamps    = np.clip(t_stamps, 10, T_MAX_SYNC - 10)
    x_scan  = np.linspace(X_NORM_108, X_NORM_107, 40)
    x_m     = x_scan * X_MAX
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    fig.suptitle("Spatial FoS profile along slope x=[7, 25] m at selected times",
                 fontweight="bold", fontsize=13)
    cmap = plt.cm.coolwarm_r
    for i, (ax, t_h) in enumerate(zip(axes.flat, t_stamps)):
        x_t = torch.tensor(x_scan, dtype=torch.float32, device=device).unsqueeze(1)
        z_t = torch.ones_like(x_t)
        t_t = torch.full_like(x_t, t_h / T_MAX_TRAIN)
        with torch.no_grad():
            _, _, fos_scan = model(x_t, z_t, t_t)
        fos_np     = fos_scan.cpu().numpy().flatten()
        rain_at_t  = float(np.interp(t_h, t_rain_h, q_rain_mmhr))
        fos_sat_f, _ = _fos_analytical_floor(0.5)
        col = cmap(np.clip((5.0 - fos_np.mean()) / 4.0, 0, 1))
        ax.plot(x_m, fos_np, color=col, lw=2.0)
        ax.fill_between(x_m, fos_np, 1.0, where=fos_np < 1.5, alpha=0.2, color="orange")
        ax.fill_between(x_m, fos_np, 1.0, where=fos_np < 1.0, alpha=0.3, color="red")
        ax.axhline(1.5, color="orange", ls="--", lw=1.2)
        ax.axhline(1.0, color="red",    ls="--", lw=1.8)
        ax.axhline(fos_sat_f, color="gray", ls=":", lw=1.0,
                   label=f"Sat. floor={fos_sat_f:.2f}")
        ax.axvline(SITE[108]["x_pos"], color="#D45F5F", lw=1.0, ls=":", label="Dev108 (x=7m)")
        ax.axvline(SITE[107]["x_pos"], color="#4A90D9", lw=1.0, ls=":", label="Dev107 (x=25m)")
        ax.scatter([SITE[108]["x_pos"], SITE[107]["x_pos"]],
                   [float(np.interp(SITE[108]["x_pos"], x_m, fos_np)),
                    float(np.interp(SITE[107]["x_pos"], x_m, fos_np))],
                   s=60, zorder=5, color=["#D45F5F","#4A90D9"], marker="D")
        # Mark split position on time axis title
        t_frac = t_h / T_MAX_SYNC
        split_tag = ("TRAIN" if t_frac < VAL_LO
                     else ("VAL" if t_frac < VAL_HI else "TEST"))
        ax.set_xlim(x_m[0]-0.5, x_m[-1]+0.5)
        ax.set_ylim(0.5, min(fos_np.max()+0.5, 8.0))
        ax.set_xlabel("x (m along slope)"); ax.set_ylabel("FoS")
        ax.set_title(f"t={t_h:.0f}h [{split_tag}]  rain={rain_at_t:.2f} mm/hr\n"
                     f"FoS∈[{fos_np.min():.2f},{fos_np.max():.2f}]", fontsize=9)
        ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    _save(fig, "fig7_spatial_fos.png")


def _fig8_baseline_comparison(model, data_loader):
    df108 = data_loader.df108
    t108_h, pred_pinn = _pred_ts(model, df108, X_NORM_108, data_loader.z_norm_108)
    obs108 = df108["theta"].values
    tr108, val108, te108 = _splits(t108_h)
    pred_pers = _persistence_pred(obs108, t108_h, window_h=24)
    pred_lstm = _lstm_pred(obs108)

    fig, axes = plt.subplots(3, 2, figsize=(14, 12))
    fig.suptitle(
        "Forecast comparison: PINN vs baselines (Dev108)\n"
        "[80/10/10 chronological split — test is strictly unseen]",
        fontweight="bold", fontsize=13)
    methods = [
        ("PINN (this work)", pred_pinn, "#E91E63"),
        ("24 h persistence", pred_pers, "#FF9800"),
        ("LSTM (data-only)", pred_lstm, "#4CAF50"),
    ]
    split_defs = [("Train (80%)", tr108), ("Val (10%)", val108),
                  ("Test (10%)\nunseen", te108)]
    for mi, (mname, pred, col) in enumerate(methods):
        ax = axes[0, min(mi, 1)]
        if mi == 2:
            ax2_twin = axes[0, 1].twinx()
            ax2_twin.plot(t108_h, pred, color=col, lw=1.0, alpha=0.6, ls=":", label=mname)
            ax2_twin.set_ylabel("LSTM θ", color=col, fontsize=8); continue
        ax.plot(t108_h, obs108, color="#333", lw=0.8, alpha=0.8, label="Observed")
        ax.plot(t108_h, pred,   color=col,   lw=1.3, ls="--", label=mname)
        ax.axvspan(T_MAX_SYNC * VAL_LO, T_MAX_SYNC * VAL_HI, alpha=0.10, color="#FF9800",
                   label="Val")
        ax.axvspan(T_MAX_SYNC * TE_LO,  T_MAX_SYNC,          alpha=0.08, color="#E91E63",
                   label="Test (unseen)")
        ax.set_title(mname, fontsize=10); ax.set_xlabel("Time (h)")
        ax.set_ylabel("θ (m³/m³)"); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

    ax_bar = axes[1, 0]
    bar_w  = 0.22; x_pos = np.arange(3)
    for mi, (mname, pred, col) in enumerate(methods):
        r2s = [_r2(obs108[mask], pred[mask]) if mask.sum() > 2 else float("nan")
               for _, mask in split_defs]
        ax_bar.bar(x_pos + mi*bar_w, r2s, width=bar_w, color=col, alpha=0.8, label=mname)
    ax_bar.set_xticks(x_pos + bar_w)
    ax_bar.set_xticklabels([s[0] for s in split_defs], fontsize=9)
    ax_bar.axhline(0.55, color="red", ls="--", lw=1.2, label="R²=0.55 threshold")
    ax_bar.set_ylabel("R²"); ax_bar.set_title("R² by split and method", fontsize=10)
    ax_bar.legend(fontsize=7); ax_bar.grid(True, alpha=0.3, axis="y")

    ax_rm  = axes[1, 1]
    for mi, (mname, pred, col) in enumerate(methods):
        rms = [_rmse(obs108[mask], pred[mask]) if mask.sum() > 2 else float("nan")
               for _, mask in split_defs]
        ax_rm.bar(x_pos + mi*bar_w, rms, width=bar_w, color=col, alpha=0.8, label=mname)
    ax_rm.set_xticks(x_pos + bar_w)
    ax_rm.set_xticklabels([s[0] for s in split_defs], fontsize=9)
    ax_rm.axhline(0.03, color="red", ls="--", lw=1.2, label="RMSE=0.03 threshold")
    ax_rm.set_ylabel("RMSE (m³/m³)")
    ax_rm.set_title("RMSE by split and method", fontsize=10)
    ax_rm.legend(fontsize=7); ax_rm.grid(True, alpha=0.3, axis="y")

    ax_sk  = axes[2, 0]
    skills_test = []
    for mname, pred, col in methods:
        if te108.sum() > 2:
            rm_model = _rmse(obs108[te108], pred[te108])
            rm_pers  = _persistence_24h_rmse(obs108[te108], t108_h[te108])
            sk = float(np.clip(1.0 - rm_model / max(rm_pers, 1e-6), -2, 2))
        else:
            sk = float("nan")
        skills_test.append((mname, sk, col))
    names_sk = [s[0] for s in skills_test]
    vals_sk  = [s[1] for s in skills_test]
    cols_sk  = [s[2] for s in skills_test]
    bars = ax_sk.bar(names_sk, vals_sk, color=cols_sk, alpha=0.8)
    ax_sk.axhline(0, color="k", lw=1.0)
    ax_sk.axhline(1, color="green", ls=":", lw=1.0, alpha=0.6)
    for bar, v in zip(bars, vals_sk):
        if np.isfinite(v):
            ax_sk.text(bar.get_x()+bar.get_width()/2, v+0.03,
                       f"{v:+.3f}", ha="center", fontsize=9)
    ax_sk.set_ylabel("Skill vs 24h persistence (test split — unseen)")
    ax_sk.set_title("Skill score — test set (strictly unseen)", fontsize=10)
    ax_sk.set_ylim(-1.5, 1.5); ax_sk.grid(True, alpha=0.3, axis="y")

    ax_sc  = axes[2, 1]
    for mname, pred, col in methods:
        if te108.sum() > 2:
            ax_sc.scatter(obs108[te108], pred[te108], s=8, alpha=0.5,
                          color=col, label=mname)
    mn = obs108[te108].min()-0.005 if te108.any() else 0.08
    mx = obs108[te108].max()+0.005 if te108.any() else 0.39
    ax_sc.plot([mn, mx], [mn, mx], "k--", lw=1.2)
    ax_sc.set_xlabel("Observed θ"); ax_sc.set_ylabel("Predicted θ")
    ax_sc.set_title("Predicted vs observed scatter\n(test split — strictly unseen)",
                    fontsize=10)
    ax_sc.legend(fontsize=7); ax_sc.set_aspect("equal"); ax_sc.grid(True, alpha=0.3)
    plt.tight_layout()
    _save(fig, "fig8_baseline_comparison.png")


# ═══════════════════════════════════════════════════════════════════════════
#  GENERATE ALL FIGURES
# ═══════════════════════════════════════════════════════════════════════════

def generate_all_figures(model, data_loader, train_hist, all_seed_results=None):
    model.eval()
    print(f"\n{'─'*55}\n  Generating 8 figures → {_OUT}/\n{'─'*55}")
    if all_seed_results is None:
        all_seed_results = [{"seed":0, "hist":train_hist,
                             "r2_tr":0, "r2_val":0, "r2_tr_108":0,
                             "r2_tr_107":0, "r2_val_108":0, "r2_val_107":0,
                             "rmse_tr_108":0, "rmse_tr_107":0,
                             "rmse_val_108":0, "rmse_val_107":0}]
    print("[Fig 1] Convergence curves..."); _fig1_convergence(train_hist, all_seed_results)
    print("[Fig 2] θ time-series fit..."); ts_data = _fig2_theta_timeseries(model, data_loader)
    print("[Fig 3] Scatter + residuals..."); _fig3_scatter_residuals(*ts_data)
    print("[Fig 4] VG retention curves..."); _fig4_vg_curves(model)
    print("[Fig 5] FoS PRIMARY output..."); _fig5_fos_primary(model, data_loader)
    print("[Fig 6] Event-level zoom..."); _fig6_event_zoom(model, data_loader)
    print("[Fig 7] Spatial FoS profile..."); _fig7_spatial_fos(model, data_loader)
    print("[Fig 8] Baseline comparison..."); _fig8_baseline_comparison(model, data_loader)
    print(f"\n[Done] 8 figures saved to {_OUT}/")


# ═══════════════════════════════════════════════════════════════════════════
#  MAIN
# ═══════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    bar = "=" * 65
    print(f"\n{bar}")
    print("  2D PINN — Single-Layer Slope Hydrology  [v3: FoS Primary Output]")
    print(f"  Changes: 80/10/10 split | no rolling window | DRY_STEP=10 | FoS output")
    print(f"{bar}\n")

    if not os.path.exists(CSV_FILE):
        raise FileNotFoundError(f"CSV not found: {CSV_FILE}")

    data = SlopeDataLoader(CSV_FILE).load()

    print(f"\n{bar}\n  STAGE 1 — Physics Discovery (single stage, no rolling window)\n{bar}")
    best_model, best_hist, all_results = train_stage1(
        data,
        n_seeds=2, total_epochs=10000, lr=2e-4,
        lam_data=500.0, lam_richards=1.0,
        lam_bc_top=0.1, lam_bc_bot=0.1,
        lam_ic=0.0, lam_smooth=0.01,
        lam_prior=0.05, lam_psi_var=1.0,
        lam_fos_physics=0.1,          # CHG-4/5: FoS physical regularisation
        es_patience=10000, anneal_epochs=4000,
    )

    pth_s1 = "pinn_slope_v3_stage1.pth"
    torch.save({
        "state_dict":    best_model.state_dict(),
        "architecture":  {"hidden":N_HIDDEN,"width":N_WIDTH,"dropout":DROPOUT},
        "site_params":   SITE,
        "stage":         "v3_single_stage",
        "learned_vg":    best_model.get_learned_params(),
        "T_MAX_TRAIN":   T_MAX_TRAIN,
        "split":         {"VAL_LO":VAL_LO,"VAL_HI":VAL_HI,"TE_LO":TE_LO,
                          "description":"80/10/10 chronological"},
        "fixes_applied": [
            "causal_bc","train_norm","skill_score","sensor_fos_only",
            "nan_tracking","raw_rain","chronological_80_10_10",
            "FIX-A_rescaling","FIX-B_pore_pressure","FIX-C_theta_s",
            "FIX-D_interp_validated","FIX-F_geotech",
            "CHG-1_80_10_10_split","CHG-2_no_rolling_window",
            "CHG-3_dry_step_10","CHG-4_fos_primary_output",
            "CHG-5_fos_physics_regularisation",
        ],
    }, pth_s1)
    print(f"[Saved] {pth_s1}")

    import shutil as _shutil
    _shutil.copy(pth_s1, os.path.join(DRIVE_CKPT_DIR, pth_s1))
    print(f"[Drive] Copied {pth_s1} → {DRIVE_CKPT_DIR}/")

    print(f"\n{bar}\n  GENERATING FIGURES\n{bar}")
    generate_all_figures(best_model, data, best_hist, all_seed_results=all_results)

    # ── FINAL SUMMARY ─────────────────────────────────────────────────────
    print(f"\n{bar}\n  FINAL SUMMARY  [v3 — 80/10/10 | FoS Primary Output]\n{bar}")
    m = compute_metrics_combined(best_model, data)

    print(f"\n  ── θ Performance (VWC) ──────────────────────────────────")
    for sname, lbl in [("tr","Train (80%)"), ("val","Val (10%)"), ("te","Test (10%) ← UNSEEN")]:
        r2  = m.get(f"r2_{sname}",   float("nan"))
        rm  = m.get(f"rmse_{sname}", float("nan"))
        n   = m.get(f"n_{sname}",    0)
        print(f"  {lbl:28s}  R²={r2:+.4f}  RMSE={rm:.5f} m³/m³  (n={n})")
    print(f"  Per-device Test:")
    for did in [107, 108]:
        r2  = m.get(f"r2_te_{did}",   float("nan"))
        rm  = m.get(f"rmse_te_{did}", float("nan"))
        print(f"    Dev{did}: R²={r2:+.4f}  RMSE={rm:.5f}")

    print(f"\n  ── FoS (PRIMARY OUTPUT) ─────────────────────────────────")
    for sname, lbl in [("tr","Train"), ("val","Val"), ("te","Test ← UNSEEN")]:
        fmean = m.get(f"fos_mean_{sname}", float("nan"))
        fmin  = m.get(f"fos_min_{sname}",  float("nan"))
        fwarn = m.get(f"fos_warn_{sname}", 0)
        ffail = m.get(f"fos_fail_{sname}", 0)
        print(f"  {lbl:12s}  FoS_mean={fmean:.3f}  FoS_min={fmin:.3f}"
              f"  warnings(<1.5)={fwarn}  failures(<1.0)={ffail}")

    print(f"\n  ── Learned VG Parameters ───────────────────────────────")
    p = best_model.get_learned_params()
    print(f"  Dev107: α={p['alpha_107']:.3f}  n={p['n_vg_107']:.4f}"
          f"  θ_r={p['theta_r_107']:.4f}  θ_s={p['theta_s_107']:.4f}")
    print(f"  Dev108: α={p['alpha_108']:.3f}  n={p['n_vg_108']:.4f}"
          f"  θ_r={p['theta_r_108']:.4f}  θ_s={p['theta_s_108']:.4f}")
    print(bar)

[Drive] Checkpoint directory : /content/pinn_checkpoints
[Drive] Periodic interval    : every 2000 epochs (stage1)
[Device] cpu

  2D PINN — Single-Layer Slope Hydrology  [v3: FoS Primary Output]
  Changes: 80/10/10 split | no rolling window | DRY_STEP=10 | FoS output

[StratB] t=0 reset to t_orig=4.58h
[CHG-1] 80/10/10 chronological split:
        Train : [0, 225h)  —  T_MAX_TRAIN=424.5h
        Val   : [225h, 267h)  (42h window)
        Test  : [382h, 424h]  (STRICTLY UNSEEN — 42h)
[DataClean-FIX-A] Dev108: range [0.4100,0.5400] preserved. 10529 readings above θₛ_lit=0.450
[FIX-C] Dev108: θₛ_init from 99th pct = 0.5178
[DataClean-FIX-A] Dev107: range [0.2850,0.4630] preserved. 937 readings above θₛ_lit=0.420
[FIX-C] Dev107: θₛ_init from 99th pct = 0.4610
[CHG-1] Rainfall in val window (Dev108): 0.0 mm  ⚠ WARNING: val window dry
[CHG-1] Rainfall in test window (Dev108): 3036.9 mm  ✓ OK
[FIX 1/6] Causal rain function set from raw record.
[PDE] Valid collocation intervals: 1 segs  424.5